# CS781 Capstone - Phase 1: Legal LM Pretraining
## Architecture 4: Deep Decoder with Latent Attention, Gated QK-Normed Heads and Engram Memory

- **Model**: 16-layer causal decoder, d_model 640, 10 heads, SwiGLU 1712, RMSNorm pre-norm, tied embeddings
- **Attention**:
   - Multi-head Latent Attention (DeepSeek-V2): keys/values from a 320-dim normalised latent, position through a decoupled 32-dim RoPE path (our A3 Q2 MLA + A3 Q3 RoPE), plus per-head QK-Norm (Qwen3) and an elementwise sigmoid output gate (Gated Attention, Qwen)
- **Memory**: Engram (DeepSeek, 2026): hashed bigram/trigram tables at layers 2 and 8, gated into the residual stream; starts as an exact no-op (`ENGRAM = False` removes it)
- **Data**: all 4 sources, document-contained 1025-token blocks, 5% document-level validation; last 10% of updates on task-matched 512-word -> 128-word examples with loss on the continuation only
- **Training**: warmup-stable-decay schedule across resumable sessions (decay on a validation plateau or a time cap); held-out BERTScore at milestones
- Design rationale and references: the Phase 1 report

**How to run**
- Session 1: attach the tokenizer dataset, GPU T4, Internet on, `HF` secret, then *Save Version -> Save & Run All* (builds and caches the data, trains until 2h before the 12h limit).
- Session 2+: attach the previous version's output, set `RUN_MODE = "resume"`, Save & Run All.
- Checkpoints every 1000 updates (and hourly), at the cutoff and on any crash outside the optimizer step.
- Test predictions and `submission.zip` are produced in the session where training completes.

## 1. HuggingFace Login

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("HuggingFace login successful.")

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

## 3. Configuration

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import hashlib
import glob
import shutil
import numpy as np
from collections import Counter

NOTEBOOK_START = time.time()  # session clock: the platform limit counts from here

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Model: deep causal decoder with MLA (DeepSeek), QK-Norm + gated attention (Qwen), Engram memory (DeepSeek)
MODEL_DIM = 640
MODEL_LAYERS = 16
MODEL_HEADS = 10           # d_head = 64 (content part of q/k, and v)
ROPE_DIM = 32              # decoupled rotary part of each q/k head (DeepSeek ratio: half of d_head)
KV_LATENT = 320            # MLA key/value latent (d_model / 2; smaller latents lost quality at this scale)
MODEL_FF = 1712            # SwiGLU: 2/3 * 4 * 640, rounded up to a multiple of 8
MODEL_SEQ_LEN = 1024
MODEL_DROPOUT = 0.0
VOCAB_SIZE = 16384
PAD_ID, BOS_ID, EOS_ID = 0, 2, 3
IGNORE_INDEX = -100

# Engram (hashed n-gram memory). Starts as an exact no-op (zero-initialised value path); ENGRAM=False removes it.
ENGRAM = True
ENGRAM_LAYERS = (2, 8)     # added to the residual stream before attention in these blocks
ENGRAM_ORDERS = (2, 3)     # bigrams and trigrams ending at each position
ENGRAM_HEADS = 4           # hash heads per order, each with its own prime-sized table
ENGRAM_HEAD_DIM = 32       # -> memory vector of 2 * 4 * 32 = 256 per position
ENGRAM_ROWS = 81920        # ~5x vocab rows per hash head (rounded up to distinct primes)
ENGRAM_LR_MULT = 5.0       # tables: 5x LR, no weight decay (Engram paper setting)

# Data
TRAIN_TOKENS = 900_000_000  # pool of document-contained training tokens (a ~20h run reads ~600-700M, so little repetition)
VAL_TOKENS = 5_000_000
BLOCK = MODEL_SEQ_LEN + 1   # 1025 tokens -> input 1024, target 1024
VAL_DOC_PCT = 5
TASK_PROMPT_WORDS = 512
TASK_TARGET_WORDS = 128
PROMPT_MAX_TOKENS = 704     # prompt tokens kept (most recent) so prompt + 320 generated tokens fit in 1024
TARGET_MAX_TOKENS = 320
TASK_POOL_SIZE = 150_000
EVAL_SET_SIZE = 400          # held-out prompts: first 50 per source = tuning set, next 50 per source = audit set
FINAL_TASK_FRAC = 0.10      # last 10% of updates: task-matched 512 -> 128 word examples, loss on the continuation only

# Training
BATCH_SIZE = 4
GRAD_ACCUM = 8              # 32 x 1024 = 32,768 tokens per update
PEAK_LR = 6e-4
MIN_LR = 6e-5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0
# Warmup-stable-decay (WSD) schedule: the learning rate stays at its peak while validation loss is still improving.
# The final decay starts when improvement flattens (or the time cap would otherwise be exceeded); training then
# finishes after a fixed-length decay. There is no fixed training duration.
MAX_TRAIN_HOURS = 24.0         # upper bound on active training, not a target
MIN_STABLE_UPDATES = 4000      # never start the decay before this
PLATEAU_WINDOW = 3             # compare validation loss across this many checks (x VAL_EVERY_UPDATES)...
PLATEAU_REL_TOL = 0.002        # ...decay starts when it improved by less than 0.2% over that window
MAX_STABLE_UPDATES = None      # optional hard cap on the stable phase in updates (None: plateau / time decide)
DECAY_FRAC = 0.20              # decay length as a fraction of the stable phase (cooldown)
MEASURE_FROM, MEASURE_TO = 50, 300   # updates used to measure throughput (for the time cap)
SESSION_LIMIT_HOURS = 12.0
SESSION_RESERVE_HOURS = 2.0  # final eval, 1000 generations (16-layer, no KV cache) and saves need this much
CKPT_EVERY_UPDATES = 1000
CKPT_EVERY_MIN = 30
SNAPSHOT_EVERY = 3000               # fp16 weight snapshots for checkpoint selection after training...
SNAPSHOT_START = 3000               # ...from this update on (the end point is not known in advance under WSD)
VAL_EVERY_UPDATES = 1000
MAX_VAL_BATCHES = 100
EVAL_EVERY_UPDATES = 8000           # held-out BERTScore milestone every N updates (plus before the task segment and at the end)
VAL_SAMPLE_BLOCKS = 400             # fixed seeded sample of validation blocks across all sources for mid-run checks
EVAL_ENABLED = True
USE_SDPA = "auto"            # attention kernel: "auto" = PyTorch SDPA only if it matches and is faster on this GPU
LOG_EVERY = 50

# Generation
TARGET_WORDS = 128
MAX_GEN_TOKENS = MODEL_SEQ_LEN - PROMPT_MAX_TOKENS   # 320
TEMPERATURE = 0.4
TOP_P = 0.92
TOP_K = 50
REPETITION_PENALTY = 1.15

# Tokenizer (same lookup as Arch 1/2/3)
_tok_matches = glob.glob("/kaggle/input/**/legal_bpe_16384.json", recursive=True)
TOKENIZER_PATH = _tok_matches[0] if _tok_matches else "/kaggle/input/legal-bpe-tokenizer/legal_bpe_16384.json"
print(f"Found tokenizer at: {TOKENIZER_PATH}")

# Run mode: "auto" resumes if a previous Arch 4 checkpoint is attached; "resume" fails fast if none is attached.
RUN_MODE = "auto"

OUTPUT_DIR = "/kaggle/working/capstone_arch4"
INPUT_ROOT = "/kaggle/input"
LATEST_CKPT = "arch4_latest.pt"
PREV_CKPT = "arch4_latest_prev.pt"
FOUNDATION_CKPT = "arch4_before_task_segment.pt"
STABLE_END_CKPT = "arch4_stable_end.pt"     # full state at the end of the stable phase: resumable for further pretraining later
CACHE_VERSION = "arch4-v2"
BEST_PHASE1 = "arch4_best_phase1_model.pt"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def find_input(name):
    m = sorted(glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    return m[-1] if m else None

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def atomic_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, default=float)
    os.replace(tmp, path)

def session_elapsed():
    return time.time() - NOTEBOOK_START

def find_resume_dir():
    """Folder holding the most advanced Arch 4 checkpoint among this session's output and all attached inputs."""
    dirs = set()
    for name in (LATEST_CKPT, PREV_CKPT):
        if os.path.exists(os.path.join(OUTPUT_DIR, name)):
            dirs.add(OUTPUT_DIR)
        dirs.update(os.path.dirname(p) for p in glob.glob(f"{INPUT_ROOT}/**/{name}", recursive=True))
    best, best_u = None, -1
    for d in sorted(dirs):
        try:
            with open(os.path.join(d, "run_log.json")) as f:
                u = int(json.load(f)["update"])
        except Exception:
            u = 0
        if u > best_u:
            best, best_u = d, u
    return best, best_u

RESUME_DIR, RESUME_UPDATE = find_resume_dir()
if RESUME_DIR:
    print(f"Previous Arch 4 run found: {RESUME_DIR} (update {RESUME_UPDATE:,}) -> this session RESUMES it")
elif RUN_MODE == "resume":
    raise RuntimeError("RUN_MODE='resume' but no arch4_latest.pt is attached: add the previous version's output as an input")
else:
    print("No previous Arch 4 checkpoint attached -> this session STARTS A NEW RUN (and builds the data, ~1-2h)")
print("Configuration loaded.")


## 4. BPE Tokenizer (from A1, our own implementation)

In [ ]:
%%writefile bpe_tokenizer.py
"""
BPE Tokenizer  - CS781 Assignment 1 implementation, optimized for Phase 1.

Speedups over original A1 code (same algorithm):
  1. fit() uses word frequencies + incremental pair-count updates
  2. encode() uses priority-based merging + word-level cache
  3. Checkpointing: saves progress every N merges so training survives Kaggle timeouts

Research-backed defaults:
  - vocab_size=16384: optimal for 100M-scale models (Zheng et al. 2024,
    "Scaling Laws with Vocabulary"). At 100M scale, 16K minimizes training
    cost; quality spread across 8K-256K is <2% BPB.
  - Corpus sample: 5-10M words is sufficient for merge stabilization.
"""

import json
import os
import time
from collections import defaultdict


class BPETokenizer:

    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.itos.extend(range(256))
        self.eow_id = len(self.itos)  # 260
        self.itos.append("</w>")
        self.merges = []
        self.vocab_size = len(self.itos)
        self._merge_rank = {}
        self._encode_cache = {}
        self._token_bytes = None
        self._ends_word = None

    # ------------------------------------------------------------------ #
    #  Training (with checkpointing)                                      #
    # ------------------------------------------------------------------ #

    def fit(self, corpus, checkpoint_dir=None, checkpoint_every=1000):
        ckpt_path = None
        if checkpoint_dir:
            os.makedirs(checkpoint_dir, exist_ok=True)
            ckpt_path = os.path.join(checkpoint_dir, f"bpe_v{self.target_vocab_size}.ckpt.json")

        resumed_merge = 0
        word_data = None

        if ckpt_path and os.path.exists(ckpt_path):
            print(f"Found checkpoint at {ckpt_path}, resuming...")
            with open(ckpt_path, "r") as f:
                ckpt = json.load(f)
            self.merges = [((a, b), c) for (a, b), c in ckpt["merges"]]
            self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
            self.itos.extend(range(256))
            self.itos.append("</w>")
            for pair, new_id in self.merges:
                self.itos.append(pair)
            self.vocab_size = len(self.itos)
            resumed_merge = len(self.merges)
            word_data = []
            for tokens, freq in ckpt["words"]:
                word_data.append([tokens, freq])
            print(f"  Resumed after {resumed_merge} merges (vocab: {self.vocab_size})")

        if word_data is None:
            print("Counting word frequencies...")
            word_freq = {}
            for word in corpus.split():
                word_freq[word] = word_freq.get(word, 0) + 1
            print(f"  {len(word_freq):,} unique words")

            word_data = []
            for word, freq in word_freq.items():
                tokens = [b + 4 for b in word.encode("utf-8")]
                tokens.append(self.eow_id)
                word_data.append([tokens, freq])

        print("Building pair index...")
        pair_counts = defaultdict(int)
        pair_words = defaultdict(set)

        for idx, (tokens, freq) in enumerate(word_data):
            seen = defaultdict(int)
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                seen[pair] += 1
            for pair, cnt in seen.items():
                pair_counts[pair] += cnt * freq
                pair_words[pair].add(idx)

        num_merges_total = self.target_vocab_size - 261
        num_merges_left = num_merges_total - resumed_merge
        print(f"Learning {num_merges_left} merges (of {num_merges_total} total)...")
        t0 = time.time()

        for merge_i in range(num_merges_left):
            if not pair_counts:
                break

            best_pair = max(pair_counts, key=pair_counts.get)
            if pair_counts[best_pair] < 2:
                break

            new_id = len(self.itos)
            bp_a, bp_b = best_pair

            affected = list(pair_words.get(best_pair, set()))

            for idx in affected:
                tokens, freq = word_data[idx]

                old_pairs = defaultdict(int)
                for i in range(len(tokens) - 1):
                    old_pairs[(tokens[i], tokens[i + 1])] += 1
                for p, cnt in old_pairs.items():
                    pair_counts[p] -= cnt * freq
                    if pair_counts[p] <= 0:
                        pair_counts.pop(p, None)
                    pair_words[p].discard(idx)
                    if not pair_words[p]:
                        pair_words.pop(p, None)

                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == bp_a and tokens[i + 1] == bp_b:
                        new_tokens.append(new_id)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                word_data[idx][0] = new_tokens

                new_pairs = defaultdict(int)
                for i in range(len(new_tokens) - 1):
                    new_pairs[(new_tokens[i], new_tokens[i + 1])] += 1
                for p, cnt in new_pairs.items():
                    pair_counts[p] = pair_counts.get(p, 0) + cnt * freq
                    pair_words[p].add(idx)

            self.merges.append((best_pair, new_id))
            self.itos.append(best_pair)
            self.vocab_size = len(self.itos)

            done = resumed_merge + merge_i + 1
            if done % 1000 == 0:
                elapsed = time.time() - t0
                rate = (merge_i + 1) / elapsed
                eta = (num_merges_left - merge_i - 1) / rate if rate > 0 else 0
                print(f"  {done}/{num_merges_total} merges | "
                      f"{elapsed:.0f}s elapsed | "
                      f"~{eta / 60:.1f} min remaining")

            if ckpt_path and (merge_i + 1) % checkpoint_every == 0:
                self._save_checkpoint(ckpt_path, word_data)

        if ckpt_path:
            self._save_checkpoint(ckpt_path, word_data)

        self._rebuild_caches()
        total_time = time.time() - t0
        print(f"Done. Final vocab: {self.vocab_size} | Time: {total_time / 60:.1f} min")
        return self

    def _save_checkpoint(self, path, word_data):
        ckpt = {
            "merges": [[[a, b], c] for (a, b), c in self.merges],
            "words": [[tokens, freq] for tokens, freq in word_data],
        }
        with open(path, "w") as f:
            json.dump(ckpt, f)

    # ------------------------------------------------------------------ #
    #  Encoding                                                           #
    # ------------------------------------------------------------------ #

    def _rebuild_caches(self):
        self._merge_rank = {}
        for rank, ((a, b), new_id) in enumerate(self.merges):
            self._merge_rank[(a, b)] = (rank, new_id)
        self._encode_cache = {}

        self._token_bytes = {x + 4: bytes([x]) for x in range(256)}
        self._token_bytes[self.eow_id] = b""
        self._ends_word = {i + 4: False for i in range(256)}
        self._ends_word[self.eow_id] = True
        for (a, b), c in self.merges:
            self._token_bytes[c] = self._token_bytes[a] + self._token_bytes[b]
            self._ends_word[c] = self._ends_word[b]

    def _encode_word(self, word):
        tokens = [b + 4 for b in word.encode("utf-8")]
        tokens.append(self.eow_id)

        while len(tokens) >= 2:
            best_rank = float("inf")
            best_idx = -1
            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                r = self._merge_rank.get(pair)
                if r is not None and r[0] < best_rank:
                    best_rank = r[0]
                    best_idx = i

            if best_idx == -1:
                break

            _, new_id = self._merge_rank[(tokens[best_idx], tokens[best_idx + 1])]
            tokens = tokens[:best_idx] + [new_id] + tokens[best_idx + 2:]

        return tokens

    def encode(self, text):
        if not self._merge_rank and self.merges:
            self._rebuild_caches()

        result = []
        for word in text.split():
            cached = self._encode_cache.get(word)
            if cached is not None:
                result.extend(cached)
            else:
                tokens = self._encode_word(word)
                self._encode_cache[word] = tokens
                result.extend(tokens)
        return result

    # ------------------------------------------------------------------ #
    #  Decoding                                                           #
    # ------------------------------------------------------------------ #

    def decode(self, tokens):
        if self._token_bytes is None:
            self._rebuild_caches()

        parts = []
        for i, token in enumerate(tokens):
            b = self._token_bytes.get(token)
            if b is not None:
                parts.append(b)
                if self._ends_word.get(token, False) and i < len(tokens) - 1:
                    parts.append(b" ")

        return b"".join(parts).decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ #
    #  Save / Load                                                        #
    # ------------------------------------------------------------------ #

    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "target_vocab_size": self.target_vocab_size,
            "merges": [[[a, b], c] for (a, b), c in self.merges],
        }
        with open(path, "w") as f:
            json.dump(data, f)
        print(f"Tokenizer saved to {path} ({self.vocab_size} tokens)")

    @classmethod
    def load(cls, path):
        with open(path, "r") as f:
            data = json.load(f)
        tok = cls(data["target_vocab_size"])
        tok.merges = [((a, b), c) for (a, b), c in data["merges"]]
        for pair, new_id in tok.merges:
            tok.itos.append(pair)
        tok.vocab_size = len(tok.itos)
        tok._rebuild_caches()
        return tok

    @property
    def pad_token_id(self):
        return 0

    @property
    def unk_token_id(self):
        return 1

    @property
    def bos_token_id(self):
        return 2

    @property
    def eos_token_id(self):
        return 3

    def __len__(self):
        return self.vocab_size

## 5. Load Tokenizer and Build the Engram Compression Map

In [ ]:
import unicodedata
from bpe_tokenizer import BPETokenizer

tokenizer = BPETokenizer.load(TOKENIZER_PATH)
assert tokenizer.vocab_size == VOCAB_SIZE, f"expected {VOCAB_SIZE}-token BPE, got {tokenizer.vocab_size}"
print(f"Loaded tokenizer: vocab_size={tokenizer.vocab_size}")
shutil.copy(TOKENIZER_PATH, os.path.join(OUTPUT_DIR, "legal_bpe_16384.json"))

# Engram tokenizer compression: token IDs whose text is identical after NFKC + accent stripping + lowercasing
# (and which agree on ending a word) share one n-gram ID. Only the Engram hash uses this; the model's vocabulary is unchanged.
# Tokens that are incomplete UTF-8 byte pieces cannot be normalised; they keep their raw bytes as a distinct key
# (decoding them with errors="replace" would turn 134 different byte tokens into the same replacement character).
def _key(b, ends_word):
    try:
        s = b.decode("utf-8")
    except UnicodeDecodeError:
        return ("bytes", b, ends_word)
    s = unicodedata.normalize("NFKC", s)
    s = "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
    return (" ".join(s.lower().split()), ends_word)

tokenizer.decode([5])  # builds the byte tables
_keys, compress = {}, []
for t in range(VOCAB_SIZE):
    b = tokenizer._token_bytes.get(t)
    key = ("special", t) if b is None else _key(b, bool(tokenizer._ends_word.get(t, False)))
    compress.append(_keys.setdefault(key, len(_keys)))
ENGRAM_COMPRESS = torch.tensor(compress, dtype=torch.long)
# (on resume, the training cell replaces this with the map stored in the checkpoint, so tables and hashing always agree)
torch.save(ENGRAM_COMPRESS, os.path.join(OUTPUT_DIR, "arch4_engram_compress.pt"))
print(f"Engram compressed vocab: {len(_keys):,} n-gram IDs from {VOCAB_SIZE:,} tokens "
      f"({100 * (1 - len(_keys) / VOCAB_SIZE):.1f}% merged)")

test_text = "The Supreme Court of India hereby declares that s. 6(1)(a) of the Act applies."
enc = tokenizer.encode(test_text)
print(f"Encode/decode: {tokenizer.decode(enc)!r} | tokens per word {len(enc)/len(test_text.split()):.2f}")


## 6. Data: Build or Reload, with Audit

Same pipeline as Arch 3 (global dedup, hash-based document split, document-contained blocks at random offsets, word-based task pool, held-out eval set), with 1025-token blocks for a decoder.

In [ ]:
try:
    import ijson
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'ijson', '-q'], check=True)
    import ijson

import array
import gc
from huggingface_hub import hf_hub_download

TOKENIZER_MD5 = hashlib.md5(open(TOKENIZER_PATH, "rb").read()).hexdigest()
CACHE = {
    "train": "arch4v2_train_blocks.npy",
    "val": "arch4v2_val_blocks.npy",
    "task": "arch4v2_task_pool.npz",
    "eval": "arch4v2_eval_set.json",
    "meta": "arch4v2_data_meta.json",
}
def find_cached(name):
    if RESUME_DIR and os.path.exists(os.path.join(RESUME_DIR, name)):
        return os.path.join(RESUME_DIR, name)
    local = os.path.join(OUTPUT_DIR, name)
    return local if os.path.exists(local) else find_input(name)

cached = {k: find_cached(v) for k, v in CACHE.items()}
if RESUME_DIR and not all(cached.values()):
    raise RuntimeError(f"resuming from {RESUME_DIR} but its data cache is incomplete: {cached}")

if all(cached.values()):
    # Later sessions: reuse the exact data built in session 1 (same split, same blocks, same pools).
    print("Loading data cache from a previous session:")
    for k, p in cached.items():
        print(f"  {k}: {p}")
        dst = os.path.join(OUTPUT_DIR, CACHE[k])
        if os.path.abspath(p) != os.path.abspath(dst):
            shutil.copy(p, dst)  # carry the cache forward so the next session's input has it too
    train_blocks = np.load(os.path.join(OUTPUT_DIR, CACHE["train"]), mmap_mode="r")
    val_blocks = np.load(os.path.join(OUTPUT_DIR, CACHE["val"]), mmap_mode="r")
    _t = np.load(os.path.join(OUTPUT_DIR, CACHE["task"]))
    task_tokens, task_offsets, task_plen = _t["tokens"], _t["offsets"], _t["plen"]
    with open(os.path.join(OUTPUT_DIR, CACHE["eval"])) as f:
        eval_set = json.load(f)
    with open(os.path.join(OUTPUT_DIR, CACHE["meta"])) as f:
        data_meta = json.load(f)
    # the cache must match this notebook's settings, otherwise resuming would silently mix two data definitions
    expect = {"block": BLOCK, "prompt_max_tokens": PROMPT_MAX_TOKENS, "target_max_tokens": TARGET_MAX_TOKENS, "seed": SEED,
              "val_doc_pct": VAL_DOC_PCT, "cache_version": CACHE_VERSION, "train_tokens_requested": TRAIN_TOKENS,
              "eval_set_size": EVAL_SET_SIZE, "dedup": "fulltext-v2", "tokenizer_md5": TOKENIZER_MD5}
    bad = {k: (data_meta.get(k), v) for k, v in expect.items() if data_meta.get(k) != v}
    assert not bad, f"data cache built with different settings (cache, notebook): {bad}"
    assert train_blocks.dtype == np.uint16 and train_blocks.ndim == 2 and train_blocks.shape[1] == BLOCK
    assert val_blocks.dtype == np.uint16 and val_blocks.shape[1] == BLOCK and len(val_blocks) > 0
    assert task_offsets[-1] == len(task_tokens) and len(task_plen) == len(task_offsets) - 1 and len(task_plen) > 0
    assert len(eval_set) > 0
else:
    print("Building data from the official corpus (session 1).")
    rng = random.Random(SEED)
    seen = set()                       # dedup key = MD5 of first 500 chars, global across all sources
    train_tok = array.array('H')
    val_tok = array.array('H')
    task_tok = array.array('H')
    task_off = [0]
    task_pl = []
    eval_set = []
    val_doc_keys = []
    stats = {}
    tokens_per_source = TRAIN_TOKENS // len(TRAIN_FILES)   # explicit equal token budget per source
    val_per_source = VAL_TOKENS // len(TRAIN_FILES)
    task_per_source = TASK_POOL_SIZE // len(TRAIN_FILES)
    eval_per_source = EVAL_SET_SIZE // len(TRAIN_FILES) + 1
    min_words = TASK_PROMPT_WORDS + TASK_TARGET_WORDS

    def doc_blocks(toks):
        n_blocks = len(toks) // BLOCK
        if n_blocks == 0:
            return None
        offset = rng.randint(0, len(toks) - n_blocks * BLOCK)  # don't always start at the heading
        return toks[offset:offset + n_blocks * BLOCK]

    def add_blocks(dst, toks, limit):
        room = (limit - len(dst)) // BLOCK * BLOCK
        if room > 0:
            dst.extend(toks[:room])
        return min(len(toks), max(room, 0)) // BLOCK

    def word_window(words):
        start = rng.randint(0, len(words) - min_words)
        return (" ".join(words[start:start + TASK_PROMPT_WORDS]),
                " ".join(words[start + TASK_PROMPT_WORDS:start + min_words]))

    for train_file in TRAIN_FILES:
        src = train_file.split("/")[-1].replace("train_", "").replace(".json", "")
        print(f"\n--- {src} ---")
        s = stats[src] = Counter()
        blocks_per_doc = []
        file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=train_file,
                                    repo_type="dataset", token=HF_TOKEN)
        train_limit = len(train_tok) + tokens_per_source
        val_limit = len(val_tok) + val_per_source
        task_start, eval_start = len(task_pl), len(eval_set)

        with open(file_path, 'rb') as f:
            for record in ijson.items(f, 'item'):
                s["docs_scanned"] += 1
                text = record.get("text") if isinstance(record, dict) else None
                if not isinstance(text, str):
                    s["malformed_skipped"] += 1
                    continue
                key = hashlib.md5(text[:500].encode()).hexdigest()            # split key: same opening -> same partition
                full_key = hashlib.md5(" ".join(text.split()).encode()).hexdigest()   # exact duplicates of the whole text
                if full_key in seen:
                    s["dup_dropped"] += 1
                    continue
                seen.add(full_key)
                words = text.split()
                if len(words) < 50:
                    s["under_50_words_dropped"] += 1
                    continue
                s["docs_kept"] += 1
                is_val = int(key[:8], 16) % 100 < VAL_DOC_PCT

                if is_val:
                    s["val_docs"] += 1
                    val_doc_keys.append(key)
                    if len(eval_set) - eval_start < eval_per_source and len(words) >= min_words:
                        p, t = word_window(words)
                        eval_set.append({"source": src, "doc_key": key, "prompt": p, "reference": t})
                    if len(val_tok) + BLOCK <= val_limit:
                        blocks = doc_blocks(tokenizer.encode(text))
                        if blocks is not None:
                            add_blocks(val_tok, blocks, val_limit)
                else:
                    s["train_docs"] += 1
                    need_blocks = len(train_tok) + BLOCK <= train_limit
                    need_task = len(task_pl) - task_start < task_per_source and len(words) >= min_words
                    if need_blocks:
                        toks = tokenizer.encode(text)
                        s["train_doc_tokens"] += len(toks)
                        blocks = doc_blocks(toks)
                        if blocks is None:
                            s["train_docs_under_1_block"] += 1
                        else:
                            n_added = add_blocks(train_tok, blocks, train_limit)
                            blocks_per_doc.append(n_added)
                    if need_task:
                        p, t = word_window(words)
                        p_ids, t_ids = tokenizer.encode(p), tokenizer.encode(t)
                        if len(p_ids) > PROMPT_MAX_TOKENS:
                            s["task_prompt_truncated"] += 1
                            p_ids = p_ids[-PROMPT_MAX_TOKENS:]     # same left-truncation as inference
                        if len(t_ids) > TARGET_MAX_TOKENS:
                            s["task_target_overlength"] += 1
                            t_ids = t_ids[:TARGET_MAX_TOKENS]
                        task_tok.extend(p_ids)
                        task_tok.extend(t_ids)
                        task_off.append(len(task_tok))
                        task_pl.append(len(p_ids))

                done_blocks = len(train_tok) + BLOCK > train_limit and len(val_tok) + BLOCK > val_limit
                done_pools = (len(task_pl) - task_start >= task_per_source
                              and len(eval_set) - eval_start >= eval_per_source)
                if done_blocks and done_pools:
                    break
                if s["docs_scanned"] % 20000 == 0:
                    print(f"  ... {s['docs_scanned']:,} docs | train={len(train_tok):,} val={len(val_tok):,} "
                          f"task={len(task_pl)-task_start:,} eval={len(eval_set)-eval_start}")
            else:
                s["source_exhausted"] = 1   # ran out of documents before filling every quota

        bpd = np.array(blocks_per_doc) if blocks_per_doc else np.zeros(1)
        s["blocks_from_source"] = int(bpd.sum())
        s["blocks_per_doc_mean"] = float(bpd.mean())
        s["blocks_per_doc_p50"] = float(np.percentile(bpd, 50))
        s["blocks_per_doc_p90"] = float(np.percentile(bpd, 90))
        s["blocks_per_doc_max"] = int(bpd.max())
        top = np.sort(bpd)[::-1]
        s["block_share_top1pct_docs"] = float(top[:max(1, len(top) // 100)].sum() / max(1, top.sum()))
        s["task_examples"] = len(task_pl) - task_start
        s["eval_examples"] = len(eval_set) - eval_start
        gc.collect()
        print(f"  {dict(s)}")

    del seen
    train_blocks = np.frombuffer(train_tok, dtype=np.uint16).reshape(-1, BLOCK).copy()
    val_blocks = np.frombuffer(val_tok, dtype=np.uint16).reshape(-1, BLOCK).copy()
    task_tokens = np.frombuffer(task_tok, dtype=np.uint16).copy()
    task_offsets = np.array(task_off, dtype=np.int64)
    task_plen = np.array(task_pl, dtype=np.int32)
    del train_tok, val_tok, task_tok
    gc.collect()
    eval_set = eval_set[:EVAL_SET_SIZE]

    data_meta = {
        "seed": SEED, "block": BLOCK, "prompt_max_tokens": PROMPT_MAX_TOKENS, "target_max_tokens": TARGET_MAX_TOKENS, "val_doc_pct": VAL_DOC_PCT,
        "cache_version": CACHE_VERSION, "train_tokens_requested": TRAIN_TOKENS, "eval_set_size": EVAL_SET_SIZE,
        "dedup": "fulltext-v2", "tokenizer_md5": TOKENIZER_MD5,
        "train_tokens_actual": int(len(train_blocks) * BLOCK), "val_tokens_actual": int(len(val_blocks) * BLOCK),
        "split_rule": "doc is validation iff int(md5(text[:500])[:8], 16) % 100 < VAL_DOC_PCT",
        "dedup_rule": "drop exact duplicates (md5 of whitespace-normalised full text) across all four sources; "
                      "documents sharing their first 500 characters share a partition through the split key",
        "source_weighting": f"equal block-token budget per source ({tokens_per_source:,} tokens each)",
        "task_pool": f"one random 512->128 word window per training doc with >= {min_words} words, until the per-source quota",
        "stats": {k: dict(v) for k, v in stats.items()},
        "val_doc_keys": val_doc_keys,
        "eval_doc_keys": [e["doc_key"] for e in eval_set],
    }
    np.save(os.path.join(OUTPUT_DIR, CACHE["train"]), train_blocks)
    np.save(os.path.join(OUTPUT_DIR, CACHE["val"]), val_blocks)
    np.savez(os.path.join(OUTPUT_DIR, CACHE["task"]), tokens=task_tokens, offsets=task_offsets, plen=task_plen)
    with open(os.path.join(OUTPUT_DIR, CACHE["eval"]), "w") as f:
        json.dump(eval_set, f)
    with open(os.path.join(OUTPUT_DIR, CACHE["meta"]), "w") as f:
        json.dump(data_meta, f)

# ---- audit report ----
print(f"\n{'='*60}\nDATA AUDIT")
for src, s in data_meta["stats"].items():
    print(f"{src:14s} scanned={s.get('docs_scanned',0):>8,} kept={s.get('docs_kept',0):>8,} "
          f"dup={s.get('dup_dropped',0):>6,} <50w={s.get('under_50_words_dropped',0):>6,} "
          f"val_docs={s.get('val_docs',0):>6,} blocks={s.get('blocks_from_source',0):>8,} "
          f"<1block={s.get('train_docs_under_1_block',0):>7,} exhausted={bool(s.get('source_exhausted'))}")
    print(f"{'':14s} blocks/doc mean={s.get('blocks_per_doc_mean',0):.1f} p50={s.get('blocks_per_doc_p50',0):.0f} "
          f"p90={s.get('blocks_per_doc_p90',0):.0f} max={s.get('blocks_per_doc_max',0)} "
          f"top-1%-docs share={s.get('block_share_top1pct_docs',0):.1%} | task={s.get('task_examples',0):,} "
          f"(prompt trunc={s.get('task_prompt_truncated',0)}, target overlength={s.get('task_target_overlength',0)}) "
          f"eval={s.get('eval_examples',0)}")
n_task = len(task_plen)
print(f"\nTrain blocks: {len(train_blocks):,} ({len(train_blocks)*BLOCK/1e6:.0f}M tokens) | val blocks: {len(val_blocks):,}")
print(f"Task pool: {n_task:,} examples | eval set: {len(eval_set)} (validation documents only)")
print("Sampling: blocks are drawn uniformly, so a document's share is proportional to its length in blocks "
      "(long documents contribute more blocks; documents under 1025 tokens are excluded from the block pool "
      "but can appear in the task pool). Task examples: at most one per document, independent of length.")
print(f"Source weighting: {data_meta['source_weighting']}")

# Tuning set (checkpoint/decoding selection) and a separate audit set (final check only), stratified by source.
# One example per document group (the split key), so tuning and audit never share a document group.
_by_src, _groups = {}, set()
for _e in eval_set:
    if _e["doc_key"] in _groups:
        continue
    _groups.add(_e["doc_key"])
    _by_src.setdefault(_e["source"], []).append(_e)
_half = max(1, EVAL_SET_SIZE // 8)
TUNE_SET = [e for src in _by_src for e in _by_src[src][:_half]]
AUDIT_SET = [e for src in _by_src for e in _by_src[src][_half:2 * _half]]
assert not ({e["doc_key"] for e in TUNE_SET} & {e["doc_key"] for e in AUDIT_SET}), "tuning and audit share a document group"
print(f"Held-out prompts: tuning {len(TUNE_SET)} | audit {len(AUDIT_SET)} (validation documents only, disjoint document groups)")

# Fixed seeded sample of validation blocks across ALL sources (blocks are stored source by source, so taking the
# first N would measure only the first source)
VAL_IDX = np.sort(np.random.default_rng(SEED + 5).choice(len(val_blocks), min(VAL_SAMPLE_BLOCKS, len(val_blocks)), replace=False))
print(f"Mid-run validation sample: {len(VAL_IDX)} blocks drawn from all {len(val_blocks):,} validation blocks")
print(f"Actual training pool: {len(train_blocks) * BLOCK / 1e6:,.0f}M tokens (requested {TRAIN_TOKENS / 1e6:,.0f}M)")


## 7. Batches: General LM and Task-Matched

In [ ]:
tokenizer._encode_cache = {}
gc.collect()


class EpochSampler:
    """Walks a seeded permutation of item indices, reshuffling each epoch. State = (epoch, pos), so it resumes exactly."""
    def __init__(self, n, seed):
        self.n, self.seed, self.epoch, self.pos = n, seed, 0, 0
        self._perm()

    def _perm(self):
        self.perm = np.random.default_rng([self.seed, self.epoch]).permutation(self.n)

    def take(self, k):
        out = []
        while len(out) < k:
            if self.pos >= self.n:
                self.epoch += 1
                self.pos = 0
                self._perm()
            m = min(k - len(out), self.n - self.pos)
            out.extend(self.perm[self.pos:self.pos + m].tolist())
            self.pos += m
        return np.array(out, dtype=np.int64)

    def state(self):
        return {"epoch": self.epoch, "pos": self.pos}

    def load(self, st):
        self.epoch, self.pos = st["epoch"], st["pos"]
        self._perm()


block_sampler = EpochSampler(len(train_blocks), SEED + 1)
task_sampler = EpochSampler(len(task_plen), SEED + 2)


def objective_for(update_idx, final_start):
    return "task" if final_start is not None and update_idx >= final_start else "lm"


def make_microbatch(obj, blocks=None, samplers=None):
    """Returns (input ids, targets) as numpy. 'lm': a 1025-token block from one document, loss on every position.
    'task': [512-word prompt (last 704 tokens) + 128-word continuation], loss on the continuation tokens only."""
    blocks = train_blocks if blocks is None else blocks
    bs, ts = (block_sampler, task_sampler) if samplers is None else samplers
    if obj == "lm":
        blk = blocks[bs.take(BATCH_SIZE)].astype(np.int64)
        return blk[:, :-1], blk[:, 1:]
    seqs, plens = [], []
    for j in ts.take(BATCH_SIZE):
        a, z, pl = task_offsets[j], task_offsets[j + 1], task_plen[j]
        seqs.append(task_tokens[a:z].astype(np.int64))
        plens.append(int(pl))
    T = max(len(s) for s in seqs) - 1
    assert T <= MODEL_SEQ_LEN
    x = np.full((BATCH_SIZE, T), PAD_ID, dtype=np.int64)
    y = np.full((BATCH_SIZE, T), IGNORE_INDEX, dtype=np.int64)
    for i, (s, pl) in enumerate(zip(seqs, plens)):
        n = len(s) - 1
        x[i, :n] = s[:-1]
        y[i, pl - 1:n] = s[pl:]          # predict the continuation tokens only (right padding stays ignored)
    return x, y


def to_device(*arrs):
    return tuple(torch.from_numpy(np.ascontiguousarray(a)).to(device, non_blocking=True) for a in arrs)


_probe = (EpochSampler(len(train_blocks), 999), EpochSampler(len(task_plen), 999))
for _obj in ("lm", "task"):
    x, y = make_microbatch(_obj, samplers=_probe)
    print(f"{_obj:5s} input {x.shape} | loss tokens per row {(y != IGNORE_INDEX).sum(1).tolist()}")


## 8. RoPE (from A3 Q3)

In [ ]:
%%writefile rope.py
"""Rotary Positional Embeddings, implemented from scratch (from A3 Q3.1)."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"

    # this is done so that we can smoothly split our dimensions correctly the way we want to

    inverse_frequencies = torch.zeros(d_head//2) # (d_head/2,)

    for i in range(0,d_head//2,1):

        inverse_frequencies[i] = (1 / (base ** (2*i / d_head))) # theta_i for all i pairs



    phi_values = torch.zeros(max_seq_len, d_head//2)
    positions = [i for i in range(0,max_seq_len)]

    for i in range(len(positions)):
        for j in range(d_head//2):
            phi_values[i,j] = positions[i] * inverse_frequencies[j]  # where positions[i] is a scalar that is being multed with another scalar value from inmverse frequencies

    # finally makes phi_values a T x d_head/2 tensor


    angles = torch.cat([phi_values,phi_values], dim=-1)  # can do this because in the phi matrix, column 0 and column d_head/2 will have same values as they have same value of i (i.e. their pair index)

    return angles.cos(), angles.sin() # sin and cos matrices



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""

    # a 90 degree rotation of (e,a) is (-a,e)
    last_dim = x.shape[-1]
    half = last_dim // 2

    x1 = x[..., :half] # if x has dimension A,B,C,D then x1 will have dimension A,B,C,D/2 and will store values of all AxBxC configurations but only D/2 values of it... like [1,2,3,4] and then  5,6,7,8 below it, so x1 will have 1,2 and below it 5,6 # just halve it along the last dim
    x2 = x[..., half:]


    ans = torch.cat([-x2, x1], dim=-1) # x rotated by 90 degree no mattewr whgat the dimension is
    return ans


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    T, d_head = cos.shape
    cos = cos.view(1,1,T,d_head)
    sin = sin.view(1,1,T,d_head)

    x_rotated = x * cos + rotate_half(x)*sin # applying rotate_half and adding just is equivalent to rotating the entire vector by theta which is what we do in rope

    return x_rotated

## 9. Multi-head Latent Attention with QK-Norm and Output Gate

In [ ]:
%%writefile attention.py
"""Multi-head Latent Attention (DeepSeek-V2 style) with decoupled RoPE, per-head QK-Norm (Qwen3) and an
elementwise sigmoid output gate (Gated Attention, Qwen). Skeleton: our A3 Q2 MultiHeadLatentAttention."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from rope import build_rope_cache, apply_rope


class RMSNorm(nn.Module):
    """x / RMS(x) * g, computed in fp32 so x*x cannot overflow under fp16 autocast."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return xf.type_as(x) * self.weight


class MultiHeadLatentAttention(nn.Module):
    """Keys/values come from a shared low-rank latent c = RMSNorm(W_down x); position enters through a separate
    rotary part of each q/k head (one rope key shared by all heads), because RoPE cannot pass through the latent.
      q_i = [q_i^C ; RoPE(q_i^R)],  k_i = [W_UK c ; RoPE(k^R)],  v_i = W_UV c
      out = W_O ( softmax(q k^T / sqrt(d_head + d_rope)) v  *  sigmoid(W_gate x) )"""
    def __init__(self, d_model, n_heads, kv_latent_dim, max_seq_len, rope_dim=32,
                 dropout=0.0, qk_norm=True, gated=True):
        super().__init__()
        assert d_model % n_heads == 0
        assert 0 < kv_latent_dim < d_model

        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.rope_dim = rope_dim
        self.kv_latent_dim = kv_latent_dim
        self.attention_scale = 1.0 / math.sqrt(self.d_head + rope_dim)

        self.q_proj = nn.Linear(d_model, n_heads * (self.d_head + rope_dim), bias=False)   # content + rotary query
        self.kv_down_proj = nn.Linear(d_model, kv_latent_dim + rope_dim, bias=False)      # latent + shared rope key
        self.kv_norm = RMSNorm(kv_latent_dim)
        self.k_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.v_up_proj = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.d_head + rope_dim) if qk_norm else nn.Identity()
        self.k_norm = RMSNorm(self.d_head + rope_dim) if qk_norm else nn.Identity()
        self.gate_proj = nn.Linear(d_model, d_model, bias=False) if gated else None
        self.use_sdpa = False      # set by the notebook after a correctness + speed check on the actual GPU

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)
        cos, sin = build_rope_cache(max_seq_len, rope_dim)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, cache=None, pos0=0):
        """cache=None: ordinary full-sequence forward (training). With a cache dict, x holds new positions
        pos0..pos0+T-1 and their keys/values are appended to the cached ones (incremental generation)."""
        B, T, C = x.shape
        H, D, R = self.n_heads, self.d_head, self.rope_dim

        q = self.q_proj(x).view(B, T, H, D + R).transpose(1, 2)              # (B, H, T, D+R)
        z = self.kv_down_proj(x)
        c, k_rope = z[..., :self.kv_latent_dim], z[..., self.kv_latent_dim:]
        c = self.kv_norm(c)
        k = self.k_up_proj(c).view(B, T, H, D).transpose(1, 2)               # (B, H, T, D)
        v = self.v_up_proj(c).view(B, T, H, D).transpose(1, 2)
        k = torch.cat([k, k_rope.unsqueeze(1).expand(B, H, T, R)], dim=-1)   # one rope key shared by all heads

        q, k = self.q_norm(q), self.k_norm(k)                                 # QK-Norm over each full head vector
        cos, sin = self.rope_cos[pos0:pos0 + T], self.rope_sin[pos0:pos0 + T]   # absolute positions
        q = torch.cat([q[..., :D], apply_rope(q[..., D:], cos, sin)], dim=-1)
        k = torch.cat([k[..., :D], apply_rope(k[..., D:], cos, sin)], dim=-1)

        if cache is not None:
            if cache.get("k") is not None:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        Tk = k.shape[2]
        mask = self.causal_mask[pos0:pos0 + T, :Tk]                          # query i sees keys <= pos0 + i

        if self.use_sdpa:
            if pos0 == 0 and T == Tk:
                final = F.scaled_dot_product_attention(q, k, v, is_causal=True, scale=self.attention_scale)
            else:
                final = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, scale=self.attention_scale)
        else:
            scores = (q @ k.transpose(-2, -1)) * self.attention_scale
            scores = scores.masked_fill(mask == 0, float('-inf'))
            probs = torch.softmax(scores, dim=-1)
            probs = self.attn_dropout(probs)
            final = probs @ v                                                # (B, H, T, D)

        if self.gate_proj is not None:                                       # head-specific elementwise gate (G1)
            final = final * torch.sigmoid(self.gate_proj(x)).view(B, T, H, D).transpose(1, 2)
        final = final.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(final)
        out = self.resid_dropout(out)
        return out


## 10. Model: Engram Memory, Blocks and Decoder

In [ ]:
%%writefile model_arch.py
"""Arch 4: deep causal decoder. MLA attention (DeepSeek) with QK-Norm and output gating (Qwen), SwiGLU FFN,
RMSNorm pre-norm, tied embeddings, and Engram hashed n-gram memory (DeepSeek) at a few layers."""
import math
from dataclasses import dataclass, field
import torch
import torch.nn as nn
import torch.nn.functional as F
from attention import MultiHeadLatentAttention, RMSNorm


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 640
    n_layers: int = 16
    n_heads: int = 10
    kv_latent_dim: int = 320
    rope_dim: int = 32
    d_ff: int = 1712
    max_seq_len: int = 1024
    dropout: float = 0.0
    engram_layers: tuple = (2, 8)
    engram_orders: tuple = (2, 3)
    engram_heads: int = 4
    engram_head_dim: int = 32
    engram_rows: int = 81920
    engram_vocab: int = 16384      # size of the compressed n-gram vocabulary


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


def _primes_from(start, count):
    out, n = [], start
    while len(out) < count:
        n += 1
        if n > 1 and all(n % p for p in range(2, int(n ** 0.5) + 1)):
            out.append(n)
    return out


class Engram(nn.Module):
    """Hashed n-gram memory (Engram). For every position t and each n-gram order n, the compressed IDs of the last n
    tokens are hashed by K independent multiplicative-XOR hashes into prime-sized tables; the retrieved vectors are
    concatenated into e_t. A context-aware gate decides how much of the memory to add:
        k = W_K e,  v = W_V e,  alpha = sigmoid(ssqrt(<RMSNorm(h), RMSNorm(k)> / sqrt(d)))
        y = SiLU(CausalDepthwiseConv(RMSNorm(alpha v))) + alpha v,   h <- h + y
    W_V and the convolution start at zero, so the module is an exact no-op at initialisation."""
    def __init__(self, cfg: LMConfig, layer_idx: int):
        super().__init__()
        self.orders = tuple(cfg.engram_orders)
        self.heads = cfg.engram_heads
        n_tables = len(self.orders) * self.heads
        primes = _primes_from(cfg.engram_rows + 101 * layer_idx, n_tables)   # ~engram_rows rows per table
        offsets = [0]
        for p in primes[:-1]:
            offsets.append(offsets[-1] + p)
        g = torch.Generator().manual_seed(1000 + layer_idx)
        mult = torch.randint(1, 2 ** 30, (n_tables, max(self.orders)), generator=g) * 2 + 1   # odd multipliers
        self.register_buffer("primes", torch.tensor(primes, dtype=torch.long), persistent=False)
        self.register_buffer("offsets", torch.tensor(offsets, dtype=torch.long), persistent=False)
        self.register_buffer("mult", mult.long(), persistent=False)
        self.register_buffer("compress", torch.arange(cfg.vocab_size, dtype=torch.long), persistent=False)

        self.table = nn.Embedding(sum(primes), cfg.engram_head_dim)
        d_mem = n_tables * cfg.engram_head_dim
        self.w_k = nn.Linear(d_mem, cfg.d_model, bias=False)
        self.w_v = nn.Linear(d_mem, cfg.d_model, bias=False)
        self.norm_h = RMSNorm(cfg.d_model)
        self.norm_k = RMSNorm(cfg.d_model)
        self.norm_v = RMSNorm(cfg.d_model)
        self.dilation = max(self.orders)
        self.conv = nn.Conv1d(cfg.d_model, cfg.d_model, kernel_size=4, groups=cfg.d_model,
                              dilation=self.dilation, bias=False)
        self.d_model = cfg.d_model

    def hash_indices(self, ids):
        """ids (B, T) -> table row indices (B, T, n_tables). Uses only tokens at positions <= t (causal)."""
        c = self.compress[ids]
        B, T = c.shape
        shifted = [c] + [F.pad(c, (j, 0), value=0)[:, :T] for j in range(1, max(self.orders))]   # token t-j
        idx, tbl = [], 0
        for n in self.orders:
            for _ in range(self.heads):
                h = shifted[0] * self.mult[tbl, 0]
                for j in range(1, n):
                    h = torch.bitwise_xor(h, shifted[j] * self.mult[tbl, j])
                idx.append(torch.remainder(h, self.primes[tbl]) + self.offsets[tbl])
                tbl += 1
        return torch.stack(idx, dim=-1)

    def forward(self, h, ids, cache=None):
        """With a cache, ids/h hold only new positions; the last (max order - 1) token IDs and the convolution's
        input history are kept so hashing and the causal convolution see exactly what the full forward sees."""
        B, T, _ = h.shape
        keep = max(self.orders) - 1
        if cache is not None and cache.get("ids") is not None:
            full_ids = torch.cat([cache["ids"], ids], dim=1)
        else:
            full_ids = ids
        idx = self.hash_indices(full_ids)[:, -T:]
        if cache is not None:
            cache["ids"] = full_ids[:, full_ids.shape[1] - keep:] if keep > 0 else full_ids[:, :0]
        e = self.table(idx).view(B, T, -1)
        k, v = self.w_k(e), self.w_v(e)
        s = (self.norm_h(h) * self.norm_k(k)).sum(-1, keepdim=True).float() / math.sqrt(self.d_model)
        alpha = torch.sigmoid(torch.sign(s) * torch.sqrt(s.abs() + 1e-6)).type_as(v)
        u = alpha * v
        y = self.norm_v(u).transpose(1, 2)                                   # (B, d, T)
        pad = (self.conv.kernel_size[0] - 1) * self.dilation
        if cache is not None:
            hist = cache.get("conv")
            if hist is None:
                hist = y.new_zeros(B, y.shape[1], pad)
            y = torch.cat([hist.to(y.dtype), y], dim=2)
            cache["conv"] = y[:, :, -pad:]
        else:
            y = F.pad(y, (pad, 0))                                           # left pad only: causal
        y = F.silu(self.conv(y)).transpose(1, 2)
        return y + u


class Block(nn.Module):
    def __init__(self, cfg: LMConfig, layer_idx: int, use_engram: bool):
        super().__init__()
        self.engram = Engram(cfg, layer_idx) if use_engram else None
        self.ln1 = RMSNorm(cfg.d_model)
        self.attn = MultiHeadLatentAttention(cfg.d_model, cfg.n_heads, cfg.kv_latent_dim, cfg.max_seq_len,
                                             rope_dim=cfg.rope_dim, dropout=cfg.dropout)
        self.ln2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, ids, cache=None, pos0=0):
        if self.engram is not None:
            x = x + self.engram(x, ids, None if cache is None else cache.setdefault("engram", {}))
        x = x + self.attn(self.ln1(x), None if cache is None else cache.setdefault("attn", {}), pos0)
        x = x + self.ffn(self.ln2(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg, i, i in tuple(cfg.engram_layers)) for i in range(cfg.n_layers)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def set_engram_compression(self, compress):
        for b in self.blocks:
            if b.engram is not None:
                b.engram.compress.copy_(compress.to(b.engram.compress.device))

    def set_attention_backend(self, use_sdpa):
        for b in self.blocks:
            b.attn.use_sdpa = bool(use_sdpa)

    def forward(self, idx):
        x = self.tok_emb(idx)
        for block in self.blocks:
            x = block(x, idx)
        x = self.ln_f(x)
        return self.head(x)

    def step(self, idx, cache=None):
        """Incremental forward for generation: idx holds the next T tokens; returns (logits, cache).
        Start with cache=None (the prompt), then feed one token at a time."""
        if cache is None:
            cache = {"pos": 0, "layers": [dict() for _ in self.blocks]}
        pos0 = cache["pos"]
        assert pos0 + idx.shape[1] <= self.cfg.max_seq_len, "generation exceeded the trained context"
        x = self.tok_emb(idx)
        for block, lc in zip(self.blocks, cache["layers"]):
            x = block(x, idx, lc, pos0)
        cache["pos"] = pos0 + idx.shape[1]
        return self.head(self.ln_f(x)), cache


## 11. Initialise the Model

In [ ]:
import torch.nn as nn
import sys
for _m in ("model_arch", "attention", "rope"):
    sys.modules.pop(_m, None)
from model_arch import LMConfig, DecoderLM, RMSNorm, Engram

cfg = LMConfig(
    vocab_size=VOCAB_SIZE, d_model=MODEL_DIM, n_layers=MODEL_LAYERS, n_heads=MODEL_HEADS,
    kv_latent_dim=KV_LATENT, rope_dim=ROPE_DIM, d_ff=MODEL_FF, max_seq_len=MODEL_SEQ_LEN, dropout=MODEL_DROPOUT,
    engram_layers=tuple(ENGRAM_LAYERS) if ENGRAM else (), engram_orders=tuple(ENGRAM_ORDERS),
    engram_heads=ENGRAM_HEADS, engram_head_dim=ENGRAM_HEAD_DIM, engram_rows=ENGRAM_ROWS,
)

torch.manual_seed(SEED)
model = DecoderLM(cfg).to(device)          # parameters stay fp32; autocast handles fp16 compute
model.set_engram_compression(ENGRAM_COMPRESS)

def init_weights(module):
    if module is model.head:
        return  # tied to tok_emb; initialise once
    if isinstance(module, nn.Linear):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            torch.nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    elif isinstance(module, RMSNorm):
        torch.nn.init.ones_(module.weight)

model.apply(init_weights)

# GPT-2-style residual scaling (a heuristic): projections writing into the residual stream get std 0.02/sqrt(2*n_layers)
residual_std = 0.02 / math.sqrt(2 * MODEL_LAYERS)
for block in model.blocks:
    torch.nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=residual_std)
    torch.nn.init.normal_(block.ffn.down_proj.weight, mean=0.0, std=residual_std)
    if block.engram is not None:          # value path and convolution at zero: Engram starts as an exact no-op
        torch.nn.init.zeros_(block.engram.w_v.weight)
        torch.nn.init.zeros_(block.engram.conv.weight)

engram_params = sum(p.numel() for b in model.blocks if b.engram is not None for p in b.engram.parameters())
table_params = sum(b.engram.table.weight.numel() for b in model.blocks if b.engram is not None)
n_params = sum(p.numel() for p in model.parameters())
assert model.head.weight is model.tok_emb.weight
assert all(p.dtype == torch.float32 for p in model.parameters())
print(f"Model parameters: {n_params:,} ({n_params/1e6:.1f}M) | backbone {(n_params - engram_params)/1e6:.1f}M | "
      f"Engram {engram_params/1e6:.1f}M (tables {table_params/1e6:.1f}M) at layers {cfg.engram_layers}")
assert 50e6 <= n_params <= 300e6, "outside the 50-300M parameter rule"

use_amp = torch.cuda.is_available()
def amp_ctx():
    return torch.autocast(device_type="cuda" if use_amp else "cpu", dtype=torch.float16, enabled=use_amp)

def GradScaler(device_type="cuda", enabled=True):
    """torch.amp.GradScaler on PyTorch >= 2.3, torch.cuda.amp.GradScaler otherwise."""
    try:
        from torch.amp import GradScaler as _GS
        return _GS(device_type, enabled=enabled)
    except (ImportError, TypeError):
        from torch.cuda.amp import GradScaler as _GS
        return _GS(enabled=enabled)
print(f"torch {torch.__version__} | fp16 autocast: {use_amp}")


## 12. Optimizer (Engram tables in their own group)

In [ ]:
# Weight decay on 2-D weights only; Engram tables get their own group (5x LR, no weight decay, as in the Engram paper)
engram_tables = {id(b.engram.table.weight) for b in model.blocks if b.engram is not None}
decay_params, no_decay_params, table_params_list = [], [], []
for name, param in model.named_parameters():
    if id(param) in engram_tables:
        table_params_list.append(param)
    elif param.dim() >= 2:
        decay_params.append(param)
    else:
        no_decay_params.append(param)

groups = [{"params": decay_params, "weight_decay": WEIGHT_DECAY, "lr_mult": 1.0},
          {"params": no_decay_params, "weight_decay": 0.0, "lr_mult": 1.0}]
if table_params_list:
    groups.append({"params": table_params_list, "weight_decay": 0.0, "lr_mult": ENGRAM_LR_MULT})
optimizer = torch.optim.AdamW(groups, lr=PEAK_LR, betas=(0.9, 0.95), eps=1e-8)
print(f"Decay params: {sum(p.numel() for p in decay_params):,} | no-decay: {sum(p.numel() for p in no_decay_params):,} | "
      f"Engram tables (x{ENGRAM_LR_MULT} LR, no decay): {sum(p.numel() for p in table_params_list):,}")


## 13. Learning Rate Schedule

In [ ]:
assert MEASURE_TO <= WARMUP_STEPS

def lr_at(update, decay_start, planned_total):
    """Warmup-stable-decay: linear warmup to PEAK_LR, constant while the model is still improving, then a cosine
    decay to MIN_LR from decay_start to planned_total (both fixed only when the stable phase ends)."""
    if update < WARMUP_STEPS:
        return PEAK_LR * update / WARMUP_STEPS
    if decay_start is None or update < decay_start:
        return PEAK_LR
    p = min(1.0, (update - decay_start) / max(1, planned_total - decay_start))
    return MIN_LR + 0.5 * (1.0 + math.cos(math.pi * p)) * (PEAK_LR - MIN_LR)

def set_lr(lr):
    for pg in optimizer.param_groups:
        pg["lr"] = lr * pg.get("lr_mult", 1.0)

def decay_plan(decay_start):
    """Total updates and the start of the task-matched segment once the decay begins (task segment = last part of the decay)."""
    total = decay_start + max(1, int(DECAY_FRAC * decay_start))
    final_start = max(decay_start, total - int(FINAL_TASK_FRAC * total))
    return total, final_start

print(f"WSD schedule: warmup {WARMUP_STEPS} -> {PEAK_LR} constant until validation loss improves <{PLATEAU_REL_TOL:.1%} over "
      f"{PLATEAU_WINDOW} checks (min {MIN_STABLE_UPDATES} updates, cap {MAX_TRAIN_HOURS}h incl. decay) -> cosine decay to "
      f"{MIN_LR} over {DECAY_FRAC:.0%} of the stable length; last {FINAL_TASK_FRAC:.0%} of the run task-matched.")


## 14. Generation and Held-out Evaluation

In [ ]:
import torch.nn.functional as F

NEVER_GEN_T = torch.tensor([PAD_ID, 1, BOS_ID, EOS_ID], device=device)


def generate(model, tokenizer, prompt, max_tokens=MAX_GEN_TOKENS, target_words=TARGET_WORDS,
             temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K, repetition_penalty=REPETITION_PENALTY,
             seed=None, return_info=False):
    """Continuation only (never the prompt). The prompt keeps its last PROMPT_MAX_TOKENS tokens (same as the
    task-matched training data), so prompt + continuation always fits the 1024-token context without sliding.
    Stops once `target_words` complete words exist; if the token cap is hit first, returns fewer words (no filler)."""
    model.eval()
    p_ids = tokenizer.encode(prompt)
    prompt_truncated = len(p_ids) > PROMPT_MAX_TOKENS
    tokens = p_ids[-PROMPT_MAX_TOKENS:]
    budget = min(max_tokens, MODEL_SEQ_LEN - len(tokens))
    gen = None
    if temperature > 0:
        gen = torch.Generator(device=device)
        gen.manual_seed(SEED if seed is None else int(seed))
    generated, complete = [], False
    cache = None
    with torch.inference_mode():
        nxt = torch.tensor([tokens], dtype=torch.long, device=device)      # prompt first, then one token per step
        for _ in range(budget):
            with amp_ctx():
                logits, cache = model.step(nxt, cache)                       # cached keys/values + Engram state
            logits = logits[0, -1].float()
            logits[NEVER_GEN_T] = float('-inf')
            if repetition_penalty != 1.0 and generated:
                ids = torch.tensor(sorted(set(generated[-100:])), device=device)
                vals = logits[ids]
                logits[ids] = torch.where(vals > 0, vals / repetition_penalty, vals * repetition_penalty)
            if temperature == 0.0:
                tok = int(logits.argmax())
            else:
                logits = logits / temperature
                if top_k > 0:
                    kth = torch.topk(logits, min(top_k, logits.size(-1))).values[-1]
                    logits[logits < kth] = float('-inf')
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cum = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                remove = cum > top_p
                remove[1:] = remove[:-1].clone()
                remove[0] = False
                logits[sorted_idx[remove]] = float('-inf')
                tok = int(torch.multinomial(torch.softmax(logits, dim=-1), 1, generator=gen))
            tokens.append(tok)
            generated.append(tok)
            nxt = torch.tensor([[tok]], dtype=torch.long, device=device)
            n_words = len(tokenizer.decode(generated).split())
            if n_words > target_words or (n_words == target_words and tokenizer._ends_word.get(tok, False)):
                complete = True
                break
    words = tokenizer.decode(generated).split()[:target_words]
    text = " ".join(words)
    info = {"n_words": len(words), "n_tokens": len(generated), "hit_token_cap": not complete,
            "prompt_truncated": prompt_truncated}
    return (text, info) if return_info else text


def val_loss(max_batches=None):
    """Next-token loss on validation blocks: a fixed seeded sample across all sources (max_batches) or all blocks."""
    was_training = model.training
    model.eval()
    idx = np.arange(len(val_blocks)) if max_batches is None else VAL_IDX[:max_batches * BATCH_SIZE]
    tot, ntok = 0.0, 0
    with torch.no_grad():
        for a in range(0, len(idx), BATCH_SIZE):
            blk = val_blocks[idx[a:a + BATCH_SIZE]].astype(np.int64)
            x, y = to_device(blk[:, :-1], blk[:, 1:])
            with amp_ctx():
                lg = model(x)
            tot += F.cross_entropy(lg.float().view(-1, lg.size(-1)), y.reshape(-1), reduction="sum").item()
            ntok += y.numel()
    if was_training:
        model.train()
    return tot / max(1, ntok)


def repetition_rate(text, n=4):
    w = text.split()
    grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return 0.0 if not grams else 1.0 - len(set(grams)) / len(grams)


def evaluate_heldout(tag):
    """Generate for the fixed held-out set (validation documents only) and score with BERTScore
    (bert_score package, roberta-large defaults: an approximation of the official scorer)."""
    from bert_score import BERTScorer   # installed and test-loaded in the pre-flight cell
    t0 = time.time()
    preds, infos, errors = [], [], []
    for i, ex in enumerate(TUNE_SET):
        try:
            text, info = generate(model, tokenizer, ex["prompt"], seed=SEED + i, return_info=True)
        except Exception as e:
            text, info = "", {"n_words": 0, "n_tokens": 0, "hit_token_cap": False, "prompt_truncated": False}
            errors.append({"idx": i, "error": repr(e)})
        preds.append(text)
        infos.append(info)
    scorer = BERTScorer(model_type="roberta-large", lang="en", device=str(device))
    P, R, F1 = scorer.score(preds, [ex["reference"] for ex in TUNE_SET])
    del scorer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    vl = val_loss(MAX_VAL_BATCHES)
    res = {"tag": tag, "update": run["update"], "bertscore_P": P.mean().item(), "bertscore_R": R.mean().item(),
           "bertscore_F1": F1.mean().item(), "val_loss": vl, "val_ppl": math.exp(min(vl, 20)),
           "mean_words": float(np.mean([x["n_words"] for x in infos])),
           "short_outputs": sum(x["n_words"] < TARGET_WORDS for x in infos),
           "token_cap_hits": sum(x["hit_token_cap"] for x in infos),
           "prompt_truncated": sum(x["prompt_truncated"] for x in infos),
           "repetition_4gram": float(np.mean([repetition_rate(p) for p in preds])),
           "generation_errors": len(errors), "n": len(TUNE_SET), "seconds": time.time() - t0,
           "per_source_F1": {s: float(np.mean([F1[i].item() for i, ex in enumerate(TUNE_SET) if ex["source"] == s]))
                             for s in sorted({ex["source"] for ex in TUNE_SET})}}
    with open(os.path.join(OUTPUT_DIR, f"eval_{tag}.json"), "w") as f:
        json.dump({"result": res, "errors": errors, "predictions": preds}, f)
    print(f"  >> EVAL [{tag}] update {res['update']}: BERTScore F1={res['bertscore_F1']:.4f} "
          f"(P={res['bertscore_P']:.4f} R={res['bertscore_R']:.4f}) | val loss={vl:.4f} | short={res['short_outputs']} "
          f"cap_hits={res['token_cap_hits']} rep4={res['repetition_4gram']:.3f} errors={len(errors)} | {res['seconds']/60:.1f} min")
    model.train()
    return res


## 15. Pre-flight Checks

In [ ]:
import copy

def check(name, cond, detail=""):
    print(f"[{'PASS' if cond else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))
    assert cond, name

print(f"Running pre-flight checks on {device} (fp16 autocast: {use_amp})\n")

# 1. LM batch: target = input shifted by one, from a single document block
s_a, s_b = EpochSampler(len(train_blocks), 4242), EpochSampler(len(train_blocks), 4242)
x, y = make_microbatch("lm", samplers=(s_a, EpochSampler(1, 0)))
blk = train_blocks[s_b.take(BATCH_SIZE)].astype(np.int64)
check("lm batch: input = block[:-1], target = block[1:]", np.array_equal(x, blk[:, :-1]) and np.array_equal(y, blk[:, 1:]))

# 2. task batch: loss only on the 128-word continuation, prompt <= PROMPT_MAX_TOKENS, padding ignored
ts = EpochSampler(len(task_plen), 7); idx = EpochSampler(len(task_plen), 7).take(BATCH_SIZE)
x, y = make_microbatch("task", samplers=(EpochSampler(1, 0), ts))
ok = True
for i, j in enumerate(idx):
    a, z, pl = task_offsets[j], task_offsets[j + 1], int(task_plen[j])
    seq = task_tokens[a:z].astype(np.int64)
    loss_pos = np.where(y[i] != IGNORE_INDEX)[0]
    ok &= len(loss_pos) == len(seq) - pl and loss_pos[0] == pl - 1 and np.array_equal(y[i, loss_pos], seq[pl:])
    ok &= pl <= PROMPT_MAX_TOKENS and np.array_equal(x[i, :len(seq) - 1], seq[:-1])
check("task batch: loss on continuation tokens only, prompt within limit", bool(ok),
      f"loss tokens {(y != IGNORE_INDEX).sum(1).tolist()}")

# fp32 copy for functional checks
mc = copy.deepcopy(model).float().eval()
g = torch.Generator().manual_seed(0)
rid = lambda *s: torch.randint(4, VOCAB_SIZE, s, generator=g).to(device)

# 3. causality through attention AND Engram (n-gram hashing + causal convolution)
ids = rid(2, 64)
with torch.no_grad():
    a = mc(ids)
    ids2 = ids.clone(); ids2[:, 40:] = rid(2, 24)
    b = mc(ids2)
check("causality: logits before an edited position are unchanged (attention + Engram)",
      torch.allclose(a[:, :40], b[:, :40], atol=1e-4), f"max diff {float((a[:, :40] - b[:, :40]).abs().max()):.2e}")

# 4. Engram is an exact no-op at initialisation
if ENGRAM:
    bypass = copy.deepcopy(mc)
    for blk_ in bypass.blocks:
        blk_.engram = None
    with torch.no_grad():
        d = float((mc(ids) - bypass(ids)).abs().max())
    check("Engram starts as an exact no-op (zero value path + zero convolution)", d == 0.0, f"max diff {d:.1e}")
    e0 = next(b_.engram for b_ in mc.blocks if b_.engram is not None)
    hi = e0.hash_indices(ids)
    sizes = e0.primes[None, None, :]
    in_range = bool(((hi - e0.offsets) >= 0).all() and ((hi - e0.offsets) < sizes).all())
    rep = ids.clone(); rep[0, 10:13] = rep[0, 30:33]          # same trigram at two positions -> same trigram rows
    hr = e0.hash_indices(rep)
    n2 = ENGRAM_HEADS
    same = torch.equal(hr[0, 12, n2:], hr[0, 32, n2:])
    check("Engram hashing: indices inside each table, identical n-grams map to identical rows", in_range and same)
    del bypass

# 5. loss normalisation with unequal microbatches = mean over all valid target tokens
mc.train()
mbs = []
for j in range(2):   # LM rows with different numbers of valid (unmasked) targets per microbatch
    xx, yy = make_microbatch("lm", samplers=(EpochSampler(len(train_blocks), 11 + j), EpochSampler(1, 0)))
    mbs.append((xx[:2, :96], np.where(np.arange(96)[None, :] < np.array([[40 + 20 * j], [60]]), yy[:2, :96], IGNORE_INDEX)))
n_valid = sum(int((yy != IGNORE_INDEX).sum()) for _, yy in mbs)
assert n_valid == 40 + 60 + 60 + 60
mc.zero_grad()
for xx, yy in mbs:
    xt, yt = to_device(xx, yy)
    (F.cross_entropy(mc(xt).view(-1, VOCAB_SIZE), yt.reshape(-1), ignore_index=IGNORE_INDEX, reduction="sum") / n_valid).backward()
g_acc = {n: p.grad.clone() for n, p in mc.named_parameters() if p.grad is not None}
mc.zero_grad()
xt, yt = to_device(np.concatenate([m[0] for m in mbs]), np.concatenate([m[1] for m in mbs]))
F.cross_entropy(mc(xt).view(-1, VOCAB_SIZE), yt.reshape(-1), ignore_index=IGNORE_INDEX).backward()
gd = max(float((p.grad - g_acc[n]).abs().max()) for n, p in mc.named_parameters() if p.grad is not None)
check("accumulated loss = mean over all valid target tokens (unequal microbatches)", gd < 1e-5, f"max grad diff {gd:.2e}")

# 6. gradients reach the MLA latent path, QK-Norm, the gate and the embeddings
gn = lambda p: 0.0 if p.grad is None else float(p.grad.norm())
att = mc.blocks[0].attn
names = {"kv_down": att.kv_down_proj.weight, "k_up": att.k_up_proj.weight, "v_up": att.v_up_proj.weight,
         "q_norm": att.q_norm.weight, "kv_norm": att.kv_norm.weight, "gate": att.gate_proj.weight,
         "tok_emb (tied)": mc.tok_emb.weight}
if ENGRAM:
    names["engram w_v"] = next(b_.engram for b_ in mc.blocks if b_.engram is not None).w_v.weight
check("gradients reach MLA latent path, QK-Norm, gate, embeddings, Engram value path",
      all(gn(p) > 0 and math.isfinite(gn(p)) for p in names.values()), ", ".join(f"{k}={gn(p):.1e}" for k, p in names.items()))
del mc, g_acc

# 7. sampler resume and objective schedule
s1 = EpochSampler(1000, 5); s1.take(1500); st = s1.state(); nxt = s1.take(50)
s2 = EpochSampler(1000, 5); s2.load(st)
check("sampler state resumes to the identical next items", np.array_equal(nxt, s2.take(50)))

# 8. save/resume on a tiny model reproduces the uninterrupted trajectory (Engram included)
def tiny():
    torch.manual_seed(3)
    m = DecoderLM(LMConfig(vocab_size=VOCAB_SIZE, d_model=64, n_layers=2, n_heads=4, kv_latent_dim=32, rope_dim=8,
                           d_ff=176, max_seq_len=MODEL_SEQ_LEN, engram_layers=(1,) if ENGRAM else (),
                           engram_heads=2, engram_head_dim=8, engram_rows=997)).to(device)
    m.set_engram_compression(ENGRAM_COMPRESS)
    return m
tb = [to_device(*(lambda b: (b[:, :64], b[:, 1:65]))(train_blocks[i * 2:(i + 1) * 2].astype(np.int64))) for i in range(4)]
def tstep(m, o, b):
    o.zero_grad()
    F.cross_entropy(m(b[0]).view(-1, VOCAB_SIZE), b[1].reshape(-1)).backward()
    o.step()
mf = tiny(); of = torch.optim.AdamW(mf.parameters(), lr=1e-3)
for b in tb:
    tstep(mf, of, b)
ma = tiny(); oa = torch.optim.AdamW(ma.parameters(), lr=1e-3)
for b in tb[:2]:
    tstep(ma, oa, b)
_tmp = os.path.join(OUTPUT_DIR, "_preflight_resume.pt")
atomic_save({"m": ma.state_dict(), "o": oa.state_dict()}, _tmp)
mb = tiny(); ob = torch.optim.AdamW(mb.parameters(), lr=1e-3)
_s = torch.load(_tmp, map_location=device, weights_only=False); mb.load_state_dict(_s["m"]); ob.load_state_dict(_s["o"])
os.remove(_tmp)
for b in tb[2:]:
    tstep(mb, ob, b)
rdiff = max(float((p - q).abs().max()) for p, q in zip(mf.parameters(), mb.parameters()))
check("save/resume reproduces the uninterrupted trajectory", rdiff < 1e-5, f"max param diff {rdiff:.2e}")
del mf, ma, mb, of, oa, ob

# 9. real-size fp16 update on a copy: finite, parameters move, and Engram tables receive gradient on the next step
mc = copy.deepcopy(model).train()
oc = torch.optim.AdamW(mc.parameters(), lr=PEAK_LR)
sc = GradScaler("cuda", enabled=use_amp)
before = mc.blocks[0].attn.kv_down_proj.weight.detach().clone()
skips, losses, gnorm, table_grad = 0, [], float("nan"), 0.0
for attempt in range(8):
    oc.zero_grad(set_to_none=True)
    bx, by = make_microbatch("lm", samplers=(EpochSampler(len(train_blocks), 77 + attempt), EpochSampler(1, 0)))
    bx, by = to_device(bx, by)
    with amp_ctx():
        lg = mc(bx)
    loss = F.cross_entropy(lg.float().view(-1, VOCAB_SIZE), by.reshape(-1))
    sc.scale(loss).backward()
    sc.unscale_(oc)
    gnorm = float(torch.nn.utils.clip_grad_norm_(mc.parameters(), GRAD_CLIP))
    if ENGRAM and losses:
        tw = next(b_.engram for b_ in mc.blocks if b_.engram is not None).table.weight
        table_grad = 0.0 if tw.grad is None else float(tw.grad.norm())
    s0 = sc.get_scale(); sc.step(oc); sc.update()
    if use_amp and sc.get_scale() < s0:
        skips += 1
        continue
    losses.append(float(loss))
    if len(losses) == 2:
        break
moved = float((mc.blocks[0].attn.kv_down_proj.weight - before).abs().max())
check("real-size updates: finite loss and gradient, parameters move",
      len(losses) == 2 and all(math.isfinite(l) for l in losses) and math.isfinite(gnorm) and moved > 0,
      f"losses {[round(l, 3) for l in losses]}, grad norm {gnorm:.3f}, skipped fp16 steps {skips}, max change {moved:.2e}")
if ENGRAM:
    check("Engram tables receive gradient once the value path has moved off zero", table_grad > 0, f"table grad norm {table_grad:.2e}")
del mc, oc, sc, lg, loss
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after checks: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

# 10. cached generation reproduces the full forward (real-size model, and a small model with Engram switched on)
def _cache_diff(m, ids, split):
    with torch.no_grad():
        full = m(ids)
        lg, cache = m.step(ids[:, :split])
        outs = [lg]
        for t in range(split, ids.shape[1]):
            lg, cache = m.step(ids[:, t:t + 1], cache)
            outs.append(lg)
    return float((full - torch.cat(outs, dim=1)).abs().max())
mc = copy.deepcopy(model).float().eval()
ids = rid(2, 48)
d_real = _cache_diff(mc, ids, 20)
torch.manual_seed(5)
tm = DecoderLM(LMConfig(vocab_size=VOCAB_SIZE, d_model=64, n_layers=3, n_heads=4, kv_latent_dim=32, rope_dim=8, d_ff=176,
                        max_seq_len=MODEL_SEQ_LEN, engram_layers=(1,), engram_heads=2, engram_head_dim=8, engram_rows=997)).to(device).eval()
tm.set_engram_compression(ENGRAM_COMPRESS)
for b_ in tm.blocks:
    if b_.engram is not None:      # non-zero value path and convolution so the Engram cache is actually exercised
        torch.nn.init.normal_(b_.engram.w_v.weight, std=0.05); torch.nn.init.normal_(b_.engram.conv.weight, std=0.2)
d_engram = _cache_diff(tm, ids, 20)
check("cached generation == full forward (attention KV cache, RoPE offsets, Engram n-gram + convolution state)",
      d_real < 1e-3 and d_engram < 1e-3, f"real-size {d_real:.1e}, small model with active Engram {d_engram:.1e}")
del mc, tm

# 11. attention kernel: PyTorch SDPA is used only if it matches the hand-written attention and is faster here
ATTENTION_BACKEND = "manual"
if USE_SDPA is True:
    ATTENTION_BACKEND = "sdpa"
elif USE_SDPA == "auto" and torch.cuda.is_available():
    mb_ = copy.deepcopy(model).train()
    xb = torch.from_numpy(train_blocks[:BATCH_SIZE, :-1].astype(np.int64)).to(device)
    timings, outs_ = {}, {}
    for name, flag in (("manual", False), ("sdpa", True)):
        mb_.set_attention_backend(flag)
        with torch.no_grad(), amp_ctx():
            outs_[name] = mb_(xb).float()
        for i_ in range(4):                       # 1 warm-up + 3 timed forward/backward passes
            if i_ == 1:
                torch.cuda.synchronize(); t_ = time.time()
            mb_.zero_grad(set_to_none=True)
            with amp_ctx():
                lo = F.cross_entropy(mb_(xb).float().view(-1, VOCAB_SIZE), xb.reshape(-1))
            lo.backward()
        torch.cuda.synchronize()
        timings[name] = (time.time() - t_) / 3
    rel = float((outs_["manual"] - outs_["sdpa"]).abs().max() / outs_["manual"].abs().max())
    faster = timings["sdpa"] < 0.95 * timings["manual"]
    ATTENTION_BACKEND = "sdpa" if (rel < 2e-2 and faster) else "manual"
    print(f"[INFO] attention kernel benchmark: manual {timings['manual']:.3f}s, SDPA {timings['sdpa']:.3f}s per fwd+bwd, "
          f"max relative output difference {rel:.1e} -> using {ATTENTION_BACKEND}")
    del mb_, outs_, xb
    torch.cuda.empty_cache()
model.set_attention_backend(ATTENTION_BACKEND == "sdpa")
print(f"Attention backend: {ATTENTION_BACKEND}")

# 12. BERTScore availability (fail fast here, never in the middle of training)
try:
    try:
        import bert_score  # noqa: F401
    except ImportError:
        import subprocess
        subprocess.run(['pip', 'install', 'bert_score', '-q'], check=True)
    from bert_score import BERTScorer
    _sc = BERTScorer(model_type="roberta-large", lang="en", device=str(device))
    _p, _r, _f = _sc.score(["the court held that the appeal is dismissed"], ["the appeal was dismissed by the court"])
    del _sc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[PASS] BERTScore (roberta-large) loads and scores (sample F1 {_f.item():.3f})")
except Exception as e:
    EVAL_ENABLED = False
    print(f"[WARN] BERTScore unavailable ({e!r}); held-out evals disabled, training unaffected")
print("\nGeneration uses the KV/Engram cache verified above. The attention kernel was chosen by measurement on this GPU; "
      "FlashAttention itself needs newer GPUs than the T4, so SDPA here means PyTorch's memory-efficient or math kernel.")


## 16. Training (resumable, one optimization trajectory)

In [ ]:
import torch.nn.functional as F
from dataclasses import asdict

OBJS = ("lm", "task")

def new_run_state():
    return {
        "update": 0, "attempts": 0, "skipped_updates": 0,
        "planned_total": None, "final_start": None, "horizon_info": None,
        "tp": {"lm": []},
        "active_train_sec": 0.0, "eval_sec": 0.0,
        "updates": {o: 0 for o in OBJS}, "examples": {o: 0 for o in OBJS},
        "loss_tokens": {o: 0 for o in OBJS},
        "history": [], "val_history": [], "evals": [], "eval_failures": [], "sessions": [], "checkpoints": [],
        "pre_final_saved": False, "mid_evals_done": [], "best_phase1": None, "done": False,
        "decay_start": None, "decay_reason": None,
    }

# settings that define this optimisation trajectory: a resumed session must use exactly these
TRAIN_SETTINGS = {"ENGRAM": ENGRAM, "FINAL_TASK_FRAC": FINAL_TASK_FRAC, "BATCH_SIZE": BATCH_SIZE, "GRAD_ACCUM": GRAD_ACCUM,
                  "PEAK_LR": PEAK_LR, "MIN_LR": MIN_LR, "WARMUP_STEPS": WARMUP_STEPS, "WEIGHT_DECAY": WEIGHT_DECAY,
                  "GRAD_CLIP": GRAD_CLIP, "DECAY_FRAC": DECAY_FRAC, "ENGRAM_LR_MULT": ENGRAM_LR_MULT,
                  "MODEL_SEQ_LEN": MODEL_SEQ_LEN, "CACHE_VERSION": CACHE_VERSION, "SEED": SEED}

def save_checkpoint(name, kind):
    """Full training state. For LATEST_CKPT the previous latest is kept as PREV_CKPT; every write is atomic."""
    run["checkpoints"].append({"name": name, "kind": kind, "update": run["update"], "unix": time.time()})
    state = {
        "kind": kind, "model": model.state_dict(), "optimizer": optimizer.state_dict(), "scaler": scaler.state_dict(),
        "model_config": asdict(cfg), "run": run,
        "samplers": {"block": block_sampler.state(), "task": task_sampler.state()},
        "rng": {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
        "lr_schedule": {"warmup": WARMUP_STEPS, "peak": PEAK_LR, "min": MIN_LR, "type": "cosine",
                        "note": "LR is a pure function of (run['update'], run['planned_total'])"},
        "engram_compress": ENGRAM_COMPRESS, "tokenizer_file": "legal_bpe_16384.json",
        "data_cache_files": CACHE, "split_rule": data_meta["split_rule"], "eval_doc_keys": data_meta["eval_doc_keys"],
        "train_settings": TRAIN_SETTINGS,
    }
    path = os.path.join(OUTPUT_DIR, name)
    tmp = path + ".tmp"
    torch.save(state, tmp)
    if name == LATEST_CKPT and os.path.exists(path):
        os.replace(path, os.path.join(OUTPUT_DIR, PREV_CKPT))
    os.replace(tmp, path)
    atomic_json({k: v for k, v in run.items() if k != "tp"}, os.path.join(OUTPUT_DIR, "run_log.json"))
    print(f"  >> saved {name} at update {run['update']:,} ({kind})")

def do_eval(tag):
    """Held-out BERTScore milestone. A failure here is logged and training continues."""
    if not EVAL_ENABLED:
        print(f"  >> eval [{tag}] skipped (BERTScore unavailable this session)")
        return
    t0 = time.time()
    try:
        res = evaluate_heldout(tag)
        run["evals"].append(res)
        if run["best_phase1"] is None or res["bertscore_F1"] > run["best_phase1"]["bertscore_F1"]:
            atomic_save(model.state_dict(), os.path.join(OUTPUT_DIR, BEST_PHASE1))
            run["best_phase1"] = {"tag": tag, "update": run["update"], "bertscore_F1": res["bertscore_F1"]}
            print(f"  >> new best Phase 1 checkpoint ({tag})")
    except Exception as e:
        run["eval_failures"].append({"tag": tag, "update": run["update"], "error": repr(e)})
        print(f"  !! eval [{tag}] failed ({e!r}); training continues")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    finally:
        model.train()
        run["eval_sec"] += time.time() - t0

def start_decay(reason):
    """End the stable phase now: fix the total length and the task-matched segment, save a resumable checkpoint."""
    total, fs = decay_plan(run["update"])
    run.update(decay_start=run["update"], planned_total=total, final_start=fs, decay_reason=reason)
    save_checkpoint(STABLE_END_CKPT, "end of stable phase (resumable for further pretraining)")
    print(f"\n  >> DECAY STARTS at update {run['update']:,} ({reason}). Run ends at update {total:,}; "
          f"task-matched segment from {fs:,}.\n")

# ---------------- optimizer state / resume ----------------
scaler = GradScaler("cuda", enabled=use_amp)
run = new_run_state()

if RESUME_DIR and os.path.abspath(RESUME_DIR) != os.path.abspath(OUTPUT_DIR):
    # carry earlier artifacts into this session's output so the next session's input has everything
    for name in (FOUNDATION_CKPT, BEST_PHASE1, PREV_CKPT, STABLE_END_CKPT, "arch4_engram_compress.pt"):
        src = os.path.join(RESUME_DIR, name)
        if os.path.exists(src) and not os.path.exists(os.path.join(OUTPUT_DIR, name)):
            shutil.copy(src, os.path.join(OUTPUT_DIR, name))
    for src in glob.glob(os.path.join(RESUME_DIR, "arch4_snap_*.pt")):
        if not os.path.exists(os.path.join(OUTPUT_DIR, os.path.basename(src))):
            shutil.copy(src, OUTPUT_DIR)
    for src in glob.glob(os.path.join(RESUME_DIR, "eval_*.json")):
        if not os.path.exists(os.path.join(OUTPUT_DIR, os.path.basename(src))):
            shutil.copy(src, OUTPUT_DIR)

ck, resume_path = None, None
if RESUME_DIR:
    for name in (LATEST_CKPT, PREV_CKPT):
        p = os.path.join(RESUME_DIR, name)
        if not os.path.exists(p):
            continue
        try:
            ck, resume_path = torch.load(p, map_location=device, weights_only=False), p
            break
        except Exception as e:
            print(f"  !! could not load {p} ({e!r}); trying the fallback checkpoint")
    if ck is None:
        raise RuntimeError(f"no loadable Arch 3 checkpoint in {RESUME_DIR}")

if ck is not None:
    print(f"Resuming from {resume_path}")
    assert ck["model_config"] == asdict(cfg), "architecture config differs from the checkpoint"
    _saved = ck.get("train_settings", {})
    _diff = {k: (_saved.get(k), v) for k, v in TRAIN_SETTINGS.items() if k in _saved and _saved[k] != v}
    assert not _diff, f"training settings differ from the checkpoint (checkpoint, notebook): {_diff}; this would change the trajectory"
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    run = ck["run"]
    for k, v in new_run_state().items():     # fields added after the checkpoint was written
        run.setdefault(k, v)
    if ck.get("engram_compress") is not None:   # keep the exact map the tables were trained with
        ENGRAM_COMPRESS = ck["engram_compress"].cpu()
        model.set_engram_compression(ENGRAM_COMPRESS)
        torch.save(ENGRAM_COMPRESS, os.path.join(OUTPUT_DIR, "arch4_engram_compress.pt"))
    if run.get("attention_backend"):            # same attention kernel as earlier sessions
        ATTENTION_BACKEND = run["attention_backend"]
        model.set_attention_backend(ATTENTION_BACKEND == "sdpa")
    block_sampler.load(ck["samplers"]["block"])
    task_sampler.load(ck["samplers"]["task"])
    random.setstate(ck["rng"]["python"]); np.random.set_state(ck["rng"]["numpy"])
    torch.set_rng_state(ck["rng"]["torch"].cpu())            # RNG states must be CPU ByteTensors; map_location moved them
    if torch.cuda.is_available() and ck["rng"]["cuda"] is not None:
        torch.cuda.set_rng_state_all([s.cpu() for s in ck["rng"]["cuda"]])
    del ck
    print(f"  update {run['update']:,} / planned {run['planned_total']} | active training so far "
          f"{run['active_train_sec']/3600:.2f}h | eval {run['eval_sec']/3600:.2f}h | skipped fp16 updates {run['skipped_updates']}")
else:
    print("Starting a new run.")

run["attention_backend"] = ATTENTION_BACKEND
print(f"Attention backend for this run: {ATTENTION_BACKEND}")
run["sessions"].append({"start_unix": time.time(), "start_update": run["update"],
                        "data_ready_after_sec": session_elapsed()})
session_deadline = (SESSION_LIMIT_HOURS - SESSION_RESERVE_HOURS) * 3600
last_ckpt = time.time()
win = {o: [0.0, 0] for o in OBJS}   # running (loss sum, tokens) per objective between log lines
consecutive_skips = 0
stop_reason = None
stage = "idle"                      # "step" while parameters may be mid-update (no emergency save then)
model.train()

try:
    while True:
        stage = "idle"
        u = run["update"]
        if run["planned_total"] is not None and u >= run["planned_total"]:
            run["done"] = True
            stop_reason = "planned update horizon reached"
            break
        if session_elapsed() >= session_deadline:
            stop_reason = f"session safety cutoff ({SESSION_LIMIT_HOURS - SESSION_RESERVE_HOURS:.2f}h of session time)"
            break

        # milestones happen at update boundaries, before the update that crosses them
        if run["final_start"] is not None and u >= run["final_start"] and not run["pre_final_saved"]:
            save_checkpoint(FOUNDATION_CKPT, "general LM, before task-matched segment")
            run["pre_final_saved"] = True
            do_eval("prefinal")
            save_checkpoint(LATEST_CKPT, "latest (before task-matched segment)")
            last_ckpt = time.time()

        t0 = time.time()                    # timed from batch construction, so the horizon reflects real wall time
        stage = "batch"
        obj = objective_for(u, run["final_start"])
        micro = [make_microbatch(obj) for _ in range(GRAD_ACCUM)]
        n_valid = sum(int((mb[1] != IGNORE_INDEX).sum()) for mb in micro)
        lr = lr_at(u, run["decay_start"], run["planned_total"])
        set_lr(lr)

        stage = "forward"
        optimizer.zero_grad(set_to_none=True)
        loss_sum = 0.0
        for mb in micro:
            bx, by = to_device(*mb)
            with amp_ctx():
                logits = model(bx)
            loss_tok = F.cross_entropy(logits.float().view(-1, logits.size(-1)), by.reshape(-1),
                                       ignore_index=IGNORE_INDEX, reduction="sum")
            scaler.scale(loss_tok / n_valid).backward()     # mean over all valid target tokens of the whole update
            loss_sum += loss_tok.item()
            del logits, loss_tok
        scaler.unscale_(optimizer)
        gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP))
        scale_before = scaler.get_scale()
        stage = "step"
        scaler.step(optimizer)
        scaler.update()
        if use_amp:
            torch.cuda.synchronize()
        dt = time.time() - t0
        run["active_train_sec"] += dt
        run["attempts"] += 1

        if (use_amp and scaler.get_scale() < scale_before) or not math.isfinite(loss_sum):
            run["skipped_updates"] += 1          # fp16 overflow: no parameter change, LR schedule does not advance
            consecutive_skips += 1
            stage = "idle"
            if consecutive_skips >= 50:
                raise RuntimeError(f"{consecutive_skips} consecutive skipped updates at update {u}; training is unstable")
            continue
        consecutive_skips = 0
        run["update"] = u + 1
        stage = "idle"
        run["updates"][obj] += 1
        run["examples"][obj] += BATCH_SIZE * GRAD_ACCUM
        run["loss_tokens"][obj] += n_valid
        win[obj][0] += loss_sum; win[obj][1] += n_valid

        if MEASURE_FROM <= u < MEASURE_TO and obj in run["tp"]:
            run["tp"][obj].append(dt)
        if run["horizon_info"] is None and run["update"] >= MEASURE_TO:
            per = float(np.mean(run["tp"]["lm"]))
            run["horizon_info"] = {"s_per_update": per}
            print(f"\n  >> THROUGHPUT: {per:.2f}s/update ({BATCH_SIZE * GRAD_ACCUM * MODEL_SEQ_LEN / per / 1e3:.1f}k tokens/s). "
                  f"The time cap ({MAX_TRAIN_HOURS}h) allows at most ~{MAX_TRAIN_HOURS * 3600 / per / (1 + DECAY_FRAC):,.0f} "
                  f"stable updates; the decay starts earlier if validation loss plateaus.\n")

        # end of the stable phase: hard cap, or the time cap would be exceeded once the decay is added
        if run["decay_start"] is None and run["update"] >= WARMUP_STEPS:
            reason = None
            if MAX_STABLE_UPDATES is not None and run["update"] >= MAX_STABLE_UPDATES:
                reason = f"stable-phase cap of {MAX_STABLE_UPDATES} updates"
            elif run["active_train_sec"] * (1 + DECAY_FRAC) >= MAX_TRAIN_HOURS * 3600 * 0.97:
                reason = f"time cap: decay must start now to finish within {MAX_TRAIN_HOURS}h"
            if reason:
                start_decay(reason)

        if run["update"] % LOG_EVERY == 0:
            parts = " ".join(f"{o}={win[o][0]/win[o][1]:.3f}" for o in OBJS if win[o][1] > 0)
            eg = [b.engram.table.weight.grad for b in model.blocks if b.engram is not None and b.engram.table.weight.grad is not None]
            if eg:
                parts += f" | engram_gnorm {float(torch.sqrt(sum((g.float() ** 2).sum() for g in eg))):.2e}"
            print(f"upd {run['update']:>6,}/{run['planned_total'] or '?'} | {parts} | lr {lr:.2e} | gnorm {gnorm:.2f} | "
                  f"scale {scaler.get_scale():.0f} | skipped {run['skipped_updates']} | {dt:.2f}s/upd | "
                  f"train {run['active_train_sec']/3600:.2f}h eval {run['eval_sec']/3600:.2f}h session {session_elapsed()/3600:.2f}h")
            run["history"].append({"update": run["update"], "lr": lr, "gnorm": gnorm, "scale": scaler.get_scale(),
                                   "active_train_sec": run["active_train_sec"],
                                   **{f"loss_{o}": win[o][0] / win[o][1] for o in OBJS if win[o][1] > 0}})
            win = {o: [0.0, 0] for o in OBJS}

        if run["update"] % VAL_EVERY_UPDATES == 0:
            tv = time.time()
            vl = val_loss(MAX_VAL_BATCHES)
            run["eval_sec"] += time.time() - tv
            run["val_history"].append({"update": run["update"], "val_loss": vl})
            print(f"  >> val loss {vl:.4f} (ppl {math.exp(min(vl, 20)):.2f})")
            hist = [v["val_loss"] for v in run["val_history"]]
            if (run["decay_start"] is None and run["update"] >= MIN_STABLE_UPDATES and len(hist) > PLATEAU_WINDOW):
                rel = (hist[-PLATEAU_WINDOW - 1] - hist[-1]) / hist[-PLATEAU_WINDOW - 1]
                if rel < PLATEAU_REL_TOL:
                    start_decay(f"validation plateau: improved {rel:.2%} over the last {PLATEAU_WINDOW} checks")

        if (run["update"] % EVAL_EVERY_UPDATES == 0
                and (run["final_start"] is None or run["update"] < run["final_start"])):
            do_eval(f"u{run['update']}")

        if run["update"] % CKPT_EVERY_UPDATES == 0 or time.time() - last_ckpt >= CKPT_EVERY_MIN * 60:
            save_checkpoint(LATEST_CKPT, "latest")
            last_ckpt = time.time()

        if run["update"] % SNAPSHOT_EVERY == 0 and run["update"] >= SNAPSHOT_START:
            snap = os.path.join(OUTPUT_DIR, f"arch4_snap_{run['update']:06d}.pt")
            atomic_save({k: v.half() for k, v in model.state_dict().items()}, snap)
            print(f"  >> weight snapshot {os.path.basename(snap)}")

except BaseException as e:
    print(f"\n!! training interrupted by {type(e).__name__}: {e}")
    if stage != "step":     # parameters, optimizer and run state agree, so this is a valid resume point
        try:
            save_checkpoint(LATEST_CKPT, f"emergency save after {type(e).__name__}")
        except BaseException as e2:
            print(f"!! emergency save failed: {e2!r}")
    else:
        print("!! interrupted inside the optimizer step; keeping the last regular checkpoint")
    raise

print(f"\nStopped: {stop_reason}")
run["sessions"][-1].update({"end_unix": time.time(), "end_update": run["update"], "session_elapsed_sec": session_elapsed()})
save_checkpoint(LATEST_CKPT, "latest (end of session)")
TRAINING_COMPLETE = run["done"]

if TRAINING_COMPLETE:
    atomic_save(model.state_dict(), os.path.join(OUTPUT_DIR, "model.pt"))   # before any generation
    print("model.pt saved.")
    do_eval("final")
    final_val_loss = val_loss()
    final_ppl = math.exp(min(final_val_loss, 20))
    run["final_val_loss"] = final_val_loss
    save_checkpoint(LATEST_CKPT, "latest (training complete)")
    print(f"Final val loss {final_val_loss:.4f} (ppl {final_ppl:.2f}) | best Phase 1 checkpoint: {run['best_phase1']}")
else:
    print(f"Training not finished: update {run['update']:,} of {run['planned_total']}. To resume: in a new version, "
          f"add this version's output as an input, set RUN_MODE = \"resume\", and Save & Run All.")
print(f"Active training {run['active_train_sec']/3600:.2f}h | eval {run['eval_sec']/3600:.2f}h | "
      f"updates per objective {run['updates']} | skipped fp16 updates {run['skipped_updates']} | "
      f"eval failures {len(run['eval_failures'])}")


## 17. Training Curves and Evaluation Summary

In [ ]:
import matplotlib.pyplot as plt

h = run["history"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for o, c in (("lm", "tab:blue"), ("task", "tab:green")):
    pts = [(x["update"], x[f"loss_{o}"]) for x in h if f"loss_{o}" in x]
    if pts:
        axes[0].plot(*zip(*pts), label=o, color=c)
if run["final_start"]:
    axes[0].axvline(run["final_start"], color="gray", ls="--", label="task-matched segment")
axes[0].set_title("Training loss per objective"); axes[0].set_xlabel("optimizer update"); axes[0].legend(); axes[0].grid(alpha=0.3)
if run["val_history"]:
    axes[1].plot([v["update"] for v in run["val_history"]], [v["val_loss"] for v in run["val_history"]], marker="o", color="tab:red")
axes[1].set_title("Validation loss"); axes[1].set_xlabel("optimizer update"); axes[1].grid(alpha=0.3)
axes[2].plot([x["update"] for x in h], [x["lr"] for x in h])
axes[2].set_title("Learning rate"); axes[2].set_xlabel("optimizer update"); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "loss_curves.png"), dpi=150)
plt.show()
for e in run["evals"]:
    print(f"eval {e['tag']:>10s} @ {e['update']:>6,}: F1 {e['bertscore_F1']:.4f} | val loss {e['val_loss']:.4f} | "
          f"rep4 {e['repetition_4gram']:.3f} | short {e['short_outputs']} | cap hits {e['token_cap_hits']}")


## 18. Load Test Data

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN,
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Number of test examples: {len(test_data)}")

# show a sample prompt
sample_id = list(test_data.keys())[0]
print(f"\nSample prompt (id={sample_id}):")
print(test_data[sample_id]["text"][:300] + "...")

## 19. Generate Test Predictions (only when training is complete)

In [ ]:
predictions = {}
gen_report = Counter()
gen_errors = []
if not TRAINING_COMPLETE:
    print("Training is not complete; skipping test generation this session.")
else:
    _best = run.get("best_phase1")
    if _best and _best["tag"] != "final" and os.path.exists(os.path.join(OUTPUT_DIR, BEST_PHASE1)):
        model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, BEST_PHASE1), map_location=device))
        print(f"Using the best checkpoint by held-out tuning F1: {_best}")
    else:
        print(f"Using the final model (best by held-out tuning F1: {_best})")
    items = list(test_data.items())
    print(f"Generating {len(items)} predictions...")
    t0 = time.time()
    for i, (example_id, record) in enumerate(items):
        try:
            text, info = generate(model, tokenizer, record["text"], seed=SEED + 100000 + i, return_info=True)
        except Exception as e:     # report it; never fabricate filler
            text, info = "", {"n_words": 0, "hit_token_cap": False, "prompt_truncated": False}
            gen_errors.append({"id": example_id, "error": repr(e)})
        predictions[str(example_id)] = {"text": text}
        gen_report["short_outputs"] += info["n_words"] < TARGET_WORDS
        gen_report["token_cap_hits"] += info["hit_token_cap"]
        gen_report["prompt_truncated"] += info["prompt_truncated"]
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(items)} | {time.time()-t0:.0f}s")
    print(f"\nGeneration complete in {(time.time()-t0)/60:.1f} min | {dict(gen_report)} | errors: {len(gen_errors)}")
    for e in gen_errors[:10]:
        print("  ", e)


## 20. Sample Predictions

In [ ]:
for i, (eid, pred) in enumerate(list(predictions.items())[:5]):
    prompt = test_data[eid]["text"]
    print(f"\n{'='*60}")
    print(f"Example {eid}")
    print(f"PROMPT: {prompt[:200]}...")
    print(f"GENERATED ({len(pred['text'].split())} words): {pred['text'][:300]}...")


## 21. Save Predictions & Create Submission

In [ ]:
if TRAINING_COMPLETE:
    _ids = [str(k) for k in test_data.keys()]
    assert set(predictions) == set(_ids) and len(predictions) == len(_ids), "prediction IDs do not match the test file"
    assert all(isinstance(v, dict) and isinstance(v.get("text"), str) for v in predictions.values())
    PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
    SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
    with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(PREDICTION_PATH, arcname="prediction.json")
    with zipfile.ZipFile(SUBMISSION_ZIP) as _z:
        assert _z.namelist() == ["prediction.json"] and json.loads(_z.read("prediction.json")) == predictions
    print(f"submission.zip created and re-validated: {SUBMISSION_ZIP} | predictions: {len(predictions)} | "
          f"avg words {sum(len(p['text'].split()) for p in predictions.values())/len(predictions):.1f}")
else:
    print("Training is not complete; no submission this session.")


## 22. Save Metrics

In [ ]:
metrics = {
    "architecture": "deep_decoder_mla_qknorm_gated_attention_engram",
    "params": n_params, "engram_params": engram_params, "model_config": asdict(cfg),
    "components": {"attention": "MLA (latent 320, decoupled RoPE 32)", "qk_norm": True, "output_gate": "elementwise sigmoid",
                   "engram": {"enabled": ENGRAM, "layers": list(ENGRAM_LAYERS), "orders": list(ENGRAM_ORDERS),
                              "heads": ENGRAM_HEADS, "head_dim": ENGRAM_HEAD_DIM}},
    "training_complete": TRAINING_COMPLETE,
    "updates": run["update"], "planned_total": run["planned_total"], "final_start": run["final_start"],
    "horizon_info": run["horizon_info"], "skipped_updates": run["skipped_updates"],
    "updates_per_objective": run["updates"], "examples_per_objective": run["examples"], "loss_tokens": run["loss_tokens"],
    "active_train_hours": run["active_train_sec"] / 3600, "eval_hours": run["eval_sec"] / 3600,
    "sessions": run["sessions"], "evals": run["evals"], "eval_failures": run["eval_failures"], "best_phase1": run["best_phase1"],
    "val_history": run["val_history"], "final_val_loss": run.get("final_val_loss"),
    "generation_config": {"temperature": TEMPERATURE, "top_p": TOP_P, "top_k": TOP_K, "repetition_penalty": REPETITION_PENALTY},
    "test_generation_report": dict(gen_report), "test_generation_errors": gen_errors,
}
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2, default=float)
print(f"metrics.json saved | training complete: {TRAINING_COMPLETE} | update {run['update']:,}/{run['planned_total']}")
print(f"Files in {OUTPUT_DIR}: {sorted(os.listdir(OUTPUT_DIR))}")
